<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/Kalman/Example8wGOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# --- CONFIGURATION & STRUCTURAL DEFINITIONS ---
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas'
]
n_states = len(state_elements)
n_periods = 6

# Mapping Matrix H_eia (Bridges the 8 hidden basin variables to 6 state EIA indicators)
H_eia = np.array([
    #P_O  EF_O H_O  Ot_O P_G  EF_G H_G  Ot_G
    [0.65, 1.0, 0.0, 0.70, 0.0, 0.0,  0.0,  0.0],  # TX Oil Total
    [0.35, 0.0, 0.0, 0.15, 0.0, 0.0,  0.0,  0.0],  # NM Oil Total
    [0.0,  0.0, 1.0, 0.15, 0.0, 0.0,  0.0,  0.0],  # LA Oil Total
    [0.0,  0.0, 0.0, 0.0,  0.65, 1.0, 0.10, 0.60], # TX Gas Total
    [0.0,  0.0, 0.0, 0.0,  0.35, 0.0, 0.0,  0.15], # NM Gas Total
    [0.0,  0.0, 0.0, 0.0,  0.0,  0.0, 0.90, 0.25]  # LA Gas Total
])

# --- DYNAMIC MATRIX PROFILE DICTIONARIES ---
# Historical Reporting Curves mapped by Age (Age 5 = Mature history, Age 0 = Bleeding Edge)
reporting_rates_by_age = {
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95]), # 5 Mos Ago (~95%)
    4: np.array([0.95, 0.94, 0.95, 0.93, 0.95, 0.94, 0.94, 0.93]),
    3: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]),
    2: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]),
    1: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]),
    0: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50])  # Current Mo (~50%)
}

# Standard deviations for Well-level data (Spikes heavily at Age 0)
well_std_by_age = {5: 15, 4: 25, 3: 50, 2: 120, 1: 250, 0: 500}
# Standard deviations for EIA top-down lines (Relatively stable across all months)
eia_std_by_age  = {5: 40, 4: 40, 3: 40, 2: 45,  1: 50,  0: 60}

# --- GENERATE TIME HORIZON ---
np.random.seed(42)
periods = pd.date_range(start="2026-03-01", periods=n_periods, freq="MS")
ages = [5, 4, 3, 2, 1, 0] # Matching timeline age pointers

# Core production values used as the true hidden baseline
true_base_values = np.array([3500, 1200, 50, 800, 2000, 3000, 14000, 4000])

fused_history = []

# --- STEP 1: LOOP THROUGH THE 6 PERIODS ---
for i, period in enumerate(periods):
    age = ages[i]
    rates = reporting_rates_by_age[age]

    # Generate Synthetic Actuals with a slight upward drift over the 6 months
    true_vals = true_base_values * (1.0 + 0.01 * i) + np.random.normal(0, 10, n_states)

    # 1. Gather Input Stream 1: Your Forecast (Stochastic prior variance)
    forecast_vals = true_vals + np.random.normal(0, 100, n_states)

    # 2. Gather Input Stream 2: Grossed-Up Well Data (Depressed by age-based reporting curve)
    well_raw = true_vals * rates + np.random.normal(0, 15, n_states)
    well_scaled = well_raw / rates

    # 3. Gather Input Stream 3: EIA-914 Top-Down State Totals
    eia_true_totals = np.dot(H_eia, true_vals)
    eia_observed = eia_true_totals + np.random.normal(0, 15, len(eia_true_totals))

    # --- STEP 2: MULTIVARIATE FILTER EXECUTION ---
    # Set initial state matrix using your forecast stream
    x = forecast_vals.copy()
    P = np.diag([120] * n_states)**2 # Prior forecast variance

    # UPDATE PHASE A: Fusing Well Data (H is Identity matrix)
    H_w = np.eye(n_states)
    R_w = np.diag([well_std_by_age[age]] * n_states)**2

    y_w = well_scaled - np.dot(H_w, x)
    S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
    K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
    x = x + np.dot(K_w, y_w)
    P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # UPDATE PHASE B: Fusing State EIA-914 Data
    R_e = np.diag([eia_std_by_age[age]] * len(eia_observed))**2

    y_e = eia_observed - np.dot(H_eia, x)
    S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
    K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
    x = x + np.dot(K_e, y_e)

    # Store complete dataset for evaluation
    fused_history.append({
        'Period': period.strftime('%Y-%m'),
        'Age': age,
        'Permian_Oil_Forecast': forecast_vals[0],
        'Permian_Oil_Wells_Scaled': well_scaled[0],
        'Permian_Oil_Fused': x[0],
        'Permian_Oil_True': true_vals[0],
        'Haynesville_Gas_Forecast': forecast_vals[6],
        'Haynesville_Gas_Wells_Scaled': well_scaled[6],
        'Haynesville_Gas_Fused': x[6],
        'Haynesville_Gas_True': true_vals[6],
    })

# --- STEP 3: ANALYZE ERROR PROFILE ---
df_results = pd.DataFrame(fused_history)
print("=== MULTI-PERIOD DATA FUSION OVERVIEW ===")
print(df_results.round(0).to_string(index=False))


=== MULTI-PERIOD DATA FUSION OVERVIEW ===
 Period  Age  Permian_Oil_Forecast  Permian_Oil_Wells_Scaled  Permian_Oil_Fused  Permian_Oil_True  Haynesville_Gas_Forecast  Haynesville_Gas_Wells_Scaled  Haynesville_Gas_Fused  Haynesville_Gas_True
2026-03    5                3458.0                    3489.0             3490.0            3505.0                   13843.0                       14017.0                14015.0               14016.0
2026-04    4                3396.0                    3522.0             3522.0            3529.0                   13994.0                       14131.0                14133.0               14142.0
2026-05    3                3601.0                    3567.0             3569.0            3565.0                   14017.0                       14304.0                14275.0               14279.0
2026-06    2                3606.0                    3639.0             3624.0            3606.0                   14407.0                       14424.0         

The Stability Shift (Age 5 to 3): For early months like 2026-03, the well data variance (well_std_by_age) is incredibly small (15). The filter relies directly on the Permian_Oil_Wells_Scaled lines, quickly overriding structural anomalies in your base forecast.The

Bleeding-Edge Protection (Age 0): For the final period (2026-08), the raw well counts provide minimal visibility. The model identifies that well_std_by_age has surged to 500, meaning it essentially bypasses the unstable scaled well volume, anchors tightly to the EIA state total balance constraints via the mapping matrix, and blends them with your forecast to land right next to the true production line.

K_e is exactly the Kalman Gain [1] for the second phase of the data fusion process (updating the model with the top-down State EIA-914 data).In a standard sequential Kalman Filter, you compute a unique Kalman Gain vector or matrix for every distinct observation stream you feed into the model.

In the 6-period code example:
K_w is the Kalman Gain calculated for your bottom-up well-level data.
K_e is the Kalman Gain calculated for your top-down state EIA data.

How K_e Acts as an Automated Weighting ValveMathematically, the Kalman Gain acts as a weighting mechanism that evaluates the relative certainty between your model's current belief and the incoming observation.Look at how K_e changes its behavior dynamically depending on the reporting maturity (Age) of the data:At Age 0 (50% reported well data): The well data is highly uncertain (R_w is massive). Consequently, the first gain factor (K_w) drops close to zero, and the model leaves the state estimate largely unchanged. When the loop hits the EIA data phase, the system realizes the EIA line is far more reliable than the messy well records (R_e is much smaller than R_w). K_e automatically scales upward, forcing the final estimate to anchor tightly to the EIA state balance constraints.At Age 5 (95%+ reported well data): The well records are now clear, solid, and reliable (R_w is tiny). The first gain factor (K_w) scales way up, instantly pinning the state estimate to the well data. By the time the code calculates K_e, the model's updated internal variance matrix (P) has shrunk to near-zero because it already found a highly certain data match. K_e automatically dials down to near-zero, preventing the less granular EIA figures from overriding or distorting the clean well data.

 Why GOR Refines the Estimates (The Core Advantage)In oil and gas data streams, gas data and oil data do not drop at the same rate, but their physical relationship is bound by the reservoir.In the ~50% Month (Age 0): A state registry might accidentally process a massive batch of natural gas volume reports for a group of new Permian wells while their corresponding crude oil volumes are stuck in a regulatory lease-accounting backlog.Without GOR: The current model sees a massive spike in partial gas data and a drop in oil data. It might overcorrect by scaling up your gas estimate while leaving your oil estimate depressed.With GOR: The model uses the physical GOR link to say: "Wait, if gas production is actually this high, the physical GOR dictatess that oil production must be higher too, despite what the partial well files say." It will automatically lift the oil estimate to match.

In [ ]:
import numpy as np
import pandas as pd

# --- STRUCTURAL CONFIGURATION ---
# The state vector now tracks 12 variables: 4 Oil Basins, 4 Gas Basins, and 4 Basin GORs
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)
n_periods = 6

# Mapping Matrix H_eia (Bridges the 12 hidden variables to the 6 state EIA indicators)
# Columns 8-11 (the GOR states) are 0 because EIA does not report a standalone GOR number.
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70] # TX Oil Total
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15] # NM Oil Total
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15] # LA Oil Total
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60] # TX Gas Total
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15] # NM Gas Total
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25] # LA Gas Total

# --- VARIANCE PROFILES BY MONTH AGE ---
reporting_rates_by_age = {
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95]), # 5 Mos Ago (~95%)
    4: np.array([0.95, 0.94, 0.95, 0.93, 0.95, 0.94, 0.94, 0.93]),
    3: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]),
    2: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]),
    1: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]),
    0: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50])  # Current Mo (~50%)
}

well_std_by_age = {5: 15, 4: 25, 3: 50, 2: 120, 1: 250, 0: 500}
eia_std_by_age  = {5: 40, 4: 40, 3: 40, 2: 45,  1: 50,  0: 60}

# Enforce tight GOR linkage at Age 0 (std=0.1), let it breathe at Age 5 (std=50.0)
# gor_constraint_std_by_age = {5: 50.0, 4: 20.0, 3: 5.0, 2: 1.0, 1: 0.2, 0: 0.1}

# NEW LOOSENED GOR PROFILE:
gor_constraint_std_by_age = {
    5: 1_000_000.0,  # 5 Mos Ago (~95%+): Effectively infinity. Completely ignores GOR.
    4: 1_000_000.0,  # 4 Mos Ago (~95%+): Completely ignores GOR.
    3: 50_000.0,     # 3 Mos Ago (~90%+): Very loose. Lets the mature actual data dominate.
    2: 1.0,          # 2 Mos Ago (~85%): Moderate constraint.
    1: 0.2,          # 1 Mo Ago (~70%): Tight constraint.
    0: 0.1           # Current Month (~50%): Ultra-tight constraint. GOR rescues missing data.
}








# --- GENERATE DATA HORIZON ---
np.random.seed(42)
periods = pd.date_range(start="2026-03-01", periods=n_periods, freq="MS")
ages = [5, 4, 3, 2, 1, 0]

# True underlying values (Oil base, Gas base, GOR base)
true_oil_base = np.array([3500, 1200, 50, 800])
true_gor_base = np.array([2.5, 3.0, 150.0, 4.0])
true_gas_base = true_oil_base * true_gor_base

fused_history = []

for i, period in enumerate(periods):
    age = ages[i]
    rates = reporting_rates_by_age[age]

    # 1. Generate True Values for this month (with dynamic monthly variance)
    true_oil = true_oil_base * (1.0 + 0.01 * i) + np.random.normal(0, 10, 4)
    true_gor = true_gor_base + np.random.normal(0, 0.05, 4) # GOR naturally drifts slightly
    true_gas = true_oil * true_gor
    true_vector = np.concatenate([true_oil, true_gas, true_gor])

    # 2. Gather Input Stream 1: Prior Forecast
    forecast_vector = true_vector + np.random.normal(0, 100, n_states)
    forecast_vector[8:12] = true_gor + np.random.normal(0, 0.2, 4) # Forecast GOR accuracy

    # 3. Gather Input Stream 2: Grossed-Up Well Data (only targets first 8 volume elements)
    well_volumes_true = true_vector[0:8]
    well_raw = well_volumes_true * rates + np.random.normal(0, 15, 8)
    well_scaled = well_raw / rates

    # 4. Gather Input Stream 3: EIA-914 Top-Down State Totals
    eia_observed = np.dot(H_eia, true_vector) + np.random.normal(0, 15, 6)

    # --- STEP 2: KALMAN FILTER DATA FUSION WITH EXTENDED GOR LAYER ---
    x = forecast_vector.copy()
    P = np.eye(n_states) * 100**2
    P[8:12, 8:12] = np.eye(4) * 0.5**2 # Variance of GOR forecast belief

    # PHASE A: Update using Partial Basin Well-Level Data (8 volume inputs)
    H_w = np.zeros((8, n_states))
    H_w[0:8, 0:8] = np.eye(8)
    R_w = np.eye(8) * (well_std_by_age[age]**2)

    y_w = well_scaled - np.dot(H_w, x)
    S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
    K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
    x = x + np.dot(K_w, y_w)
    P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # PHASE B: Update using State EIA-914 Data (6 inputs)
    R_e = np.eye(6) * (eia_std_by_age[age]**2)

    y_e = eia_observed - np.dot(H_eia, x)
    S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
    K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
    x = x + np.dot(K_e, y_e)
    P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # PHASE C: The Fully Integrated GOR Matrix Update (4 constraints)
    # Re-linearize the physical equation around current estimates: Gas - (GOR * Oil) = 0
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        current_oil = x[b]
        current_gor = x[b+8]
        H_gor[b, b]   = -current_gor # Derivative with respect to Oil
        H_gor[b, b+4] = 1.0          # Derivative with respect to Gas
        H_gor[b, b+8] = -current_oil # Derivative with respect to GOR

    # The mathematical target for the residual equation is always 0
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))

    # Assign the dynamic age-dependent constraint variance
    R_gor = np.eye(4) * (gor_constraint_std_by_age[age]**2)

    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    fused_history.append({
        'Period': period.strftime('%Y-%m'),
        'Age': age,
        'Permian_Oil_Forecast': forecast_vector[0],
        'Permian_Oil_Wells_Scaled': well_scaled[0],
        'Permian_Oil_Fused': x[0],
        'Permian_Oil_True': true_vector[0],
        'Permian_Gas_Forecast': forecast_vector[4],
        'Permian_Gas_Wells_Scaled': well_scaled[4],
        'Permian_Gas_Fused': x[4],
        'Permian_Gas_True': true_vector[4],
        'Permian_GOR_Fused': x[8],
        'Permian_GOR_True': true_vector[8]
    })

df_results = pd.DataFrame(fused_history)
print("=== GOR-INTEGRATED MULTI-PERIOD ESTIMATION ===")
print(df_results.round(1).to_string(index=False))


=== GOR-INTEGRATED MULTI-PERIOD ESTIMATION ===
 Period  Age  Permian_Oil_Forecast  Permian_Oil_Wells_Scaled  Permian_Oil_Fused  Permian_Oil_True  Permian_Gas_Forecast  Permian_Gas_Wells_Scaled  Permian_Gas_Fused  Permian_Gas_True  Permian_GOR_Fused  Permian_GOR_True
2026-03    5                3458.0                    3496.5             3495.0            3505.0                8745.6                    8712.0             8711.5            8721.4                2.8               2.5
2026-04    4                3475.7                    3504.2             3508.6            3521.7                8816.3                    8782.8             8782.4            8783.9                2.6               2.5
2026-05    3                3490.0                    3547.3             3549.3            3570.9                8835.0                    8885.3             8870.4            8888.0                2.5               2.5
2026-06    2                3743.4                    3617.4             

Why the results are much better:Phase C is an Extended Kalman Filter (EKF) step: Because \(Gas = Oil \times GOR\) is a non-linear relationship, the loop recalculates the derivatives (H_gor) using the updated variables in real time.The Cross-Pollination Benefit: At Age 0, if your Permian_Oil_Wells_Scaled number encounters a bad reporting slump, the model relies on Phase C to look at the gas stream and the tracked GOR state to immediately force the oil estimate to correct itself.

https://www.google.com/search?q=in+some+series+we+get+delay+reporting.+but+some+data+is+already+suggesting+the+outcome.+how+to+predict+the+outcome+with+partial+data%3F&ie=UTF-8&oe=UTF-8&hl=en-us&client=safari&fbs=ABfTbFUxGEP8yeZbmk97ajdTjIq-Fell6yjIojusYtuKjXhLi43HlmHdBhkjA3l1LeWMNI31J3wjh5ota8viteVD7M7JWhWI1sJN_Y-p91HosJvxELbf3xOAD9pl09itgH0OE_NT6Mg1wyQNC-omONpSlPhrmGWkLje8lo-5dZEjjSqkJ51QKkUlCt8SiG48ELLsMYZnaxdIrMwCYEaqwpjhOooydHJfwOCThv4Z87TBXWVMdIg1kwg&aep=10&ntc=1&sxsrf=APpeQnudhjQ4f-WzAmQ4qemAEy0xtX6OFA:1788520140838&mstk=AUtExfDhAGI7_eoTJoQdnkqXCk1dNcX0Iufjkynm3VMli7iyaHYeV2fcCfMs_OT1TZUs8uDn2e1mukzCbiBz6wSxfK1CZp0Dkch_ch_lckW0lcQoS4a_T9TL1IfxMHWMvLvU3_gZNfrnQR_wD6qSfs-Cp3lQbm8K0k3O35AsT3aIjd4W2DcJnuwKCkX6u73pSrRIAL56woHhUldGmq0CiJfITSKHmBDIN4xBlvn5LtNXz2wPsKTmYIRC1ceLx37RhoYaIIpTMpaw8zjzYZ1ynKqWBsolM_GAnDzuijhGGaYw7z0UyAfWxYjuzwjPRHEtgMvHsET-cJ6f8ZMFYsttujNBbz4BMrZD3kxi3Q&csuir=1&aioh=3&atvm=2&mtid=QaiaatT8BuznwbkPquf_uA8&udm=50&iga=1&utm_campaign=safari_share_1

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND PREPARE THE RAW DATA
# ==========================================
# Read the CSV. Assumes first column contains dates (e.g., '2024-01-01')
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
df.rename(columns={df.columns[0]: 'Month'}, inplace=True)

# Define our 12 internal hidden state variables
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# Set up the EIA-914 top-down state mapping matrix (H_eia)
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70]   # TX Total Oil allocation
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15]   # NM Total Oil allocation
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15]   # LA Total Oil allocation
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60]  # TX Total Gas allocation
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15]  # NM Total Gas allocation
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25]  # LA Total Gas allocation

# ==========================================
# 2. DEFINE DYNAMIC PARAMETERS BY MONTH AGE
# ==========================================
# Age is determined by months lagging behind June 2026
# June 2026 = Age 0, May 2026 = Age 1, April 2026 = Age 2, etc. (Age 5+ is mature history)
reporting_rates_by_age = {
    0: np.array([1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5, 1e-5]), # Age 0: June 2026 (No regional data yet)
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]), # Age 1: May 2026 (~50% reported well data)
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]), # Age 2: April 2026 (~70%)
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]), # Age 3: March 2026 (~85%)
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]), # Age 4: Feb 2026 (~90%)
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])  # Age 5+: History (~95%+)
}

# Volume standard deviations (Spikes for younger regional data)
well_std_by_age = {0: 1e9, 1: 500, 2: 250, 3: 120, 4: 50, 5: 15}
eia_std_by_age  = {0: 60,  1: 60,  2: 50,  3: 45,  4: 40, 5: 40}

# GOR constraint loosening rules (Ultra-tight at Age 1, disabled at Age 5+)
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1000000.0, 5: 1000000.0}

# ==========================================
# 3. PROCESSING LOOP THROUGH FILE TIMELINE
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

# Sort timeline chronologically starting Jan 2024
df = df.sort_values('Month').reset_index(drop=True)

for idx, row in df.iterrows():
    month_dt = row['Month']

    # Calculate month delta/age relative to current benchmark (June 2026)
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5) # Cap profile lookup at 5 for historical entries

    # Extract structural state inputs from CSV rows
    # Expected columns: o_TX, o_NM, o_LA, g_TX, g_NM, g_LA
    eia_observed = np.array([
        row.get('o_TX', 0), row.get('o_NM', 0), row.get('o_LA', 0),
        row.get('g_TX', 0), row.get('g_NM', 0), row.get('g_LA', 0)
    ])

    # Extract bottom-up regional volumes from CSV rows
    # Expected columns: o_Permian, o_EagleFord, o_Haynesville, o_Other, etc.
    well_raw = np.array([
        row.get('o_Permian', 0), row.get('o_EagleFord', 0), row.get('o_Haynesville', 0), row.get('o_Other', 0),
        row.get('g_Permian', 0), row.get('g_EagleFord', 0), row.get('g_Haynesville', 0), row.get('g_Other', 0)
    ])

    # Gross-up the raw well volumes using the target age's expected curve
    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # ------------------------------------------
    # INITIALIZE STATE TRACKER PRIORS (Forecast Step)
    # ------------------------------------------
    # For this script, we'll auto-seed the prior forecast loop from your data baseline,
    # plus an initial expected GOR fallback profile if corporate forecasts aren't mapped.
    initial_gor_guess = np.array([2.5, 3.0, 150.0, 4.0])

    if well_scaled.sum() > 0:
        x = np.concatenate([well_scaled[0:4], well_scaled[4:8], initial_gor_guess])
    else:
        # Fallback for June 2026 (Age 0) where we only have state data
        oil_seed = np.array([eia_observed[0]*0.5, eia_observed[0]*0.3, eia_observed[2], eia_observed[0]*0.2])
        gas_seed = oil_seed * initial_gor_guess
        x = np.concatenate([oil_seed, gas_seed, initial_gor_guess])

    P = np.eye(n_states) * 150**2
    P[8:12, 8:12] = np.eye(4) * 0.5**2 # GOR track variance

    # ------------------------------------------
    # PHASE A: FUSE REGIONAL WELL DATA (If Available)
    # ------------------------------------------
    if age > 0: # Skip Phase A for June 2026 since well data doesn't exist yet
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # ------------------------------------------
    # PHASE B: FUSE STATE EIA-914 DATA
    # ------------------------------------------
    R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)

    y_e = eia_observed - np.dot(H_eia, x)
    S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
    K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
    x = x + np.dot(K_e, y_e)
    P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # ------------------------------------------
    # PHASE C: EXTENDED GOR FILTER LAYER
    # ------------------------------------------
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8] # d/dOil
        H_gor[b, b+4] = 1.0     # d/dGas
        H_gor[b, b+8] = -x[b]   # d/dGOR

    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)

    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # Save the output arrays
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3], # Oil
        x[4], x[5], x[6], x[7], # Gas
        x[8], x[9], x[10], x[11] # GOR
    ])

# ==========================================
# 4. OUTPUT RESULTS
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== CALCULATED RESULTS FOR RECENT TIMELINE ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== CALCULATED RESULTS FOR RECENT TIMELINE ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                    0.0                      0.0                        0.0                  0.0                    0.0                      0.0                        0.0                  0.0                    2.5                      3.0                      150.0                  4.0
2026-02    4                    0.0                      0.0                        0.0                  0.0                    0.0                      0.0                        0.0                  0.0                    2.5                      3.0                      150.0                  4.0
2026-03    3                    0.0               

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND PREPARE THE RAW DATA
# ==========================================
df = pd.read_csv('TX2LA.csv')

# Force the first column to be the Date index
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Standardize column headers to make the script robust to naming conventions
# This converts headers like "o_TX", "o TX", or "oTX" into "otx"
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

# Helper function to safely fetch columns using flexible naming rules
def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return row.get(clean_name, 0)

# Define our 12 internal hidden state variables
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# Set up the EIA-914 top-down state mapping matrix (H_eia)
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70]   # TX Total Oil allocation
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15]   # NM Total Oil allocation
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15]   # LA Total Oil allocation
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60]  # TX Total Gas allocation
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15]  # NM Total Gas allocation
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25]  # LA Total Gas allocation

# ==========================================
# 2. DEFINE DYNAMIC PARAMETERS BY MONTH AGE
# ==========================================
reporting_rates_by_age = {
    0: np.array([1e-5]*8), # June 2026 (No regional data yet)
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]), # May 2026 (~50%)
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]), # April 2026 (~70%)
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]), # March 2026 (~85%)
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]), # Feb 2026 (~90%)
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])  # History (~95%+)
}

well_std_by_age = {0: 1e6, 1: 500, 2: 250, 3: 120, 4: 50, 5: 15}
eia_std_by_age  = {0: 60,  1: 60,  2: 50,  3: 45,  4: 40, 5: 40}
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1e6, 5: 1e6}

# ==========================================
# 3. PROCESSING LOOP THROUGH FILE TIMELINE
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]

    # Calculate exact month age relative to June 2026
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5)

    # Extract structural state inputs (EIA)
    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    # Extract bottom-up regional volumes
    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    # Gross-up the raw well volumes
    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # Initialize state tracker priors
    initial_gor_guess = np.array([2.5, 3.0, 150.0, 4.0])

    if well_scaled.sum() > 0:
        x = np.concatenate([well_scaled[0:4], well_scaled[4:8], initial_gor_guess])
    else:
        # Fixed fallback parsing for June 2026 (Age 0) where regional data is missing
        tx_oil, nm_oil, la_oil = eia_observed[0], eia_observed[1], eia_observed[2]
        tx_gas, nm_gas, la_gas = eia_observed[3], eia_observed[4], eia_observed[5]

        # Approximate a baseline distribution to seed the filter
        oil_seed = np.array([tx_oil*0.5 + nm_oil*0.7, tx_oil*0.3, la_oil*0.8, tx_oil*0.2])
        gas_seed = oil_seed * initial_gor_guess
        x = np.concatenate([oil_seed, gas_seed, initial_gor_guess])

    P = np.eye(n_states) * 500**2
    P[8:12, 8:12] = np.eye(4) * 1.0**2

    # PHASE A: FUSE REGIONAL WELL DATA
    if age > 0 and well_raw.sum() > 0:
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # PHASE B: FUSE STATE EIA-914 DATA
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        y_e = eia_observed - np.dot(H_eia, x)
        S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
        K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
        x = x + np.dot(K_e, y_e)
        P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # PHASE C: EXTENDED GOR FILTER LAYER
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]
        H_gor[b, b+4] = 1.0
        H_gor[b, b+8] = -x[b]

    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)

    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # FIXED: Extract individual scalar entries using explicit array indices
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 4. OUTPUT RESULTS
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== CALCULATED RESULTS FOR RECENT TIMELINE ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== CALCULATED RESULTS FOR RECENT TIMELINE ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6510.8                   1154.9                       21.4                357.5                30071.0                   8934.2                    15691.4               7684.5                    2.5                      3.0                      150.0                  4.0
2026-02    4                 7176.7                   1087.7                       30.5                221.9                33358.8                  10150.2                    13969.2               7551.0                    2.5                      3.0                      150.0                  4.0
2026-03    3                 6618.3               

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND PREPARE THE RAW DATA
# ==========================================
df = pd.read_csv('TX2LA.csv')

# Force chronological sort starting in Jan 2024
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Standardize column headers (removes spaces, underscores, lowercase)
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return row.get(clean_name, 0)

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# EIA-914 Top-Down State Mapping Matrix (H_eia)
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70]   # TX Total Oil
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15]   # NM Total Oil
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15]   # LA Total Oil
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60]  # TX Total Gas
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15]  # NM Total Gas
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25]  # LA Total Gas

# ==========================================
# 2. TIME-SERIES MODEL CONFIGURATION (MEMORY & SHOCK ABSORBERS)
# ==========================================
# F Matrix: Assumes production carries over 1:1 into next month (Random Walk Baseline)
F = np.eye(n_states)

# Q Matrix (Process Noise): Dictates maximum allowed physical variation per month.
# Lower values act as heavy dampers, completely blocking unphysical halves or spikes.
Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], 25.0**2)    # Max expected true monthly oil shift (~25 kbd)
np.fill_diagonal(Q[4:8, 4:8], 100.0**2)   # Max expected true monthly gas shift (~100 mmcfd)
np.fill_diagonal(Q[8:12, 8:12], 0.02**2)  # Max expected monthly GOR geological drift

# Reporting curves based on data age (Age 0 = June 2026)
reporting_rates_by_age = {
    0: np.array([1e-5]*8),
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]), # May 2026
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]), # Apr 2026
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]), # Mar 2026
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]), # Feb 2026
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])  # Mature History
}

# Measurement standard deviations (R)
well_std_by_age = {0: 1e6, 1: 300, 2: 150, 3: 80, 4: 40, 5: 15}
# eia_std_by_age  = {0: 40,  1: 40,  2: 35,  3: 30,  4: 30, 5: 30}
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1e6, 5: 1e6}

# NEW STRICT-CHECK PROFILE:
eia_std_by_age = {
    0: 1e-4,  # June 2026: Near-zero variance forces absolute compliance
    1: 1e-4,  # May 2026
    2: 1e-4,  # Apr 2026
    3: 1e-4,  # Mar 2026
    4: 1e-4,  # Feb 2026 (Fixes the February overflow)
    5: 1e-4   # History
}




# ==========================================
# 3. INITIALIZE THE PERSISTENT MEMORY STATES (JAN 2024 SEED)
# ==========================================
first_row = df.iloc[0]
initial_gor = np.array([2.5, 3.0, 150.0, 4.0])
initial_oil = np.array([get_val(first_row, 'o', 'Permian'), get_val(first_row, 'o', 'EagleFord'), get_val(first_row, 'o', 'Haynesville'), get_val(first_row, 'o', 'Other')])
initial_gas = np.array([get_val(first_row, 'g', 'Permian'), get_val(first_row, 'g', 'EagleFord'), get_val(first_row, 'g', 'Haynesville'), get_val(first_row, 'g', 'Other')])

# Set default values if first month data is missing/partial
if initial_oil.sum() == 0:
    initial_oil = np.array([3000, 1000, 50, 500])
    initial_gas = initial_oil * initial_gor

# x and P are declared OUTSIDE the loop. They preserve memory throughout the timeline.
x = np.concatenate([initial_oil, initial_gas, initial_gor])
P = np.eye(n_states) * 100**2

fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

# ==========================================
# 4. TIME-SERIES RECURSION LOOP
# ==========================================
for idx, row in df.iterrows():
    month_dt = row.iloc[0]

    # Calculate exact month age relative to June 2026 boundary
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5)

    # ------------------------------------------
    # MODEL STEP 1: TIME PREDICTION (Carry memory forward)
    # ------------------------------------------
    if idx > 0:
        x = np.dot(F, x)       # Project previous state estimate forward
        P = np.dot(F, np.dot(P, F.T)) + Q  # Project variance and inject process damping constraint

    # Extract row metrics
    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # ------------------------------------------
    # MODEL STEP 2: MEASUREMENT UPDATE CORES
    # ------------------------------------------
    # PHASE A: FUSE BOTTOM-UP WELL DATA
    if age > 0 and well_raw.sum() > 0:
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # PHASE B: FUSE TOP-DOWN STATE EIA-914 DATA
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        y_e = eia_observed - np.dot(H_eia, x)
        S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
        K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
        x = x + np.dot(K_e, y_e)
        P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # PHASE C: EXTENDED GOR FILTER LAYER
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]
        H_gor[b, b+4] = 1.0
        H_gor[b, b+8] = -x[b]

    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)

    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # Append scalar elements to structural results array
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 5. GENERATE FINAL REPORT
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== SMOOTHED TIME-SERIES MEMORY OUTCOMES ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== SMOOTHED TIME-SERIES MEMORY OUTCOMES ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 5948.0                   1527.9                       29.8                241.3                30270.6                  12384.8                     9782.1               7128.5                    4.4                      5.1                      150.0                  4.4
2026-02    4                 6455.8                   1414.8                       22.5                289.9                31983.2                  12472.4                    10460.2               6859.2                    4.4                      5.1                      150.0                  4.4
2026-03    3                 6481.7                 

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND PREPARE THE RAW DATA
# ==========================================
df = pd.read_csv('TX2LA.csv')

# Force chronological sort starting in Jan 2024
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Standardize column headers (removes spaces, underscores, lowercase)
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return row.get(clean_name, 0)

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# EIA-914 Top-Down State Mapping Matrix (H_eia)
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70]   # TX Total Oil
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15]   # NM Total Oil
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15]   # LA Total Oil
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60]  # TX Total Gas
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15]  # NM Total Gas
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25]  # LA Total Gas

# ==========================================
# 2. TIME-SERIES MODEL CONFIGURATION (MEMORY & SHOCK ABSORBERS)
# ==========================================
# F Matrix: Assumes production carries over 1:1 into next month (Random Walk Baseline)
F = np.eye(n_states)

# Q Matrix (Process Noise): Dictates maximum allowed physical variation per month.
# Lower values act as heavy dampers, completely blocking unphysical halves or spikes.
Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], 25.0**2)    # Max expected true monthly oil shift (~25 kbd)
np.fill_diagonal(Q[4:8, 4:8], 100.0**2)   # Max expected true monthly gas shift (~100 mmcfd)
np.fill_diagonal(Q[8:12, 8:12], 0.02**2)  # Max expected monthly GOR geological drift

# Reporting curves based on data age (Age 0 = June 2026)
reporting_rates_by_age = {
    0: np.array([1e-5]*8),
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]), # May 2026
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]), # Apr 2026
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]), # Mar 2026
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]), # Feb 2026
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])  # Mature History
}

# Measurement standard deviations (R)
well_std_by_age = {0: 1e6, 1: 300, 2: 150, 3: 80, 4: 40, 5: 15}
# eia_std_by_age  = {0: 40,  1: 40,  2: 35,  3: 30,  4: 30, 5: 30}
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1e6, 5: 1e6}

# NEW STRICT-CHECK PROFILE:
eia_std_by_age = {
    0: 1e-4,  # June 2026: Near-zero variance forces absolute compliance
    1: 1e-4,  # May 2026
    2: 1e-4,  # Apr 2026
    3: 1e-4,  # Mar 2026
    4: 1e-4,  # Feb 2026 (Fixes the February overflow)
    5: 1e-4   # History
}


# ==========================================
# [Keep Steps 1 & 2 from your previous code intact]
# ==========================================

# Make sure eia_std_by_age is tight (e.g., 1e-4) to enforce the state check
eia_std_by_age = {0: 1e-4, 1: 1e-4, 2: 1e-4, 3: 1e-4, 4: 1e-4, 5: 1e-4}

# Initialize variables outside the loop as before
# x = ...
# P = ...

fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5)

    # --- MODEL STEP 1: TIME PREDICTION ---
    if idx > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    # Extract row metrics
    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # --- PHASE A: FUSE BOTTOM-UP WELL DATA ---
    if age > 0 and well_raw.sum() > 0:
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # --- PHASE B: FUSE STATE EIA-914 DATA ---
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        y_e = eia_observed - np.dot(H_eia, x)
        S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
        K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
        x = x + np.dot(K_e, y_e)
        P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # --- PHASE C: EXTENDED GOR FILTER LAYER ---
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]
        H_gor[b, b+4] = 1.0
        H_gor[b, b+8] = -x[b]
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)
    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # ------------------------------------------
    # NEW PHASE D: ENFORCE BOUNDARY FLOOR EXTRACTION
    # ------------------------------------------
    # 1. Enforce Floor for Oil (indices 0 to 3)
    for b in range(3): # Check key basins first: Permian, EagleFord, Haynesville
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]          # Hard clip to the reported floor
            x[3] = x[3] - discrepancy   # Shift the correction to the 'Other' bucket

    # 2. Enforce Floor for Gas (indices 4 to 7)
    for b in range(4, 7):
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]          # Hard clip to the reported floor
            x[7] = x[7] - discrepancy   # Shift the correction to the 'Other' bucket

    # 3. Final Safety Net: Re-check that 'Other' bucket hasn't been pushed below its floor
    if x[3] < well_raw[3]: x[3] = well_raw[3]
    if x[7] < well_raw[7]: x[7] = well_raw[7]

    # Append records to the tracking array
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# [Keep Step 5 Result Printing Intact]
# =======================

# ==========================================
# 5. GENERATE FINAL REPORT
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== SMOOTHED TIME-SERIES MEMORY OUTCOMES ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== SMOOTHED TIME-SERIES MEMORY OUTCOMES ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6267.0                   1532.8                       31.6                351.0                30280.5                  12391.5                    15527.0               7270.0                    4.6                      8.4                      366.2                 24.4
2026-02    4                 6662.0                   1420.8                       25.0                330.0                32046.1                  12515.5                    16284.0               7190.0                    4.6                      8.4                      366.2                 24.4
2026-03    3                 6608.0                 

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
# Read file and ensure chronological sorting
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Normalize column names: strip spaces, underscores, and convert to lowercase
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return row.get(clean_name, 0)

# Define our 12 internal hidden state variables
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# EIA-914 Top-Down State Mapping Matrix (H_eia)
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70]   # TX Total Oil allocation
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15]   # NM Total Oil allocation
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15]   # LA Total Oil allocation
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60]  # TX Total Gas allocation
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15]  # NM Total Gas allocation
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25]  # LA Total Gas allocation

# ==========================================
# 2. TIME-SERIES TRACKING PARAMETERS
# ==========================================
# F Matrix: Carry state forward 1:1 into next month (Random Walk Base Memory)
F = np.eye(n_states)

# Q Matrix (Process Noise): Maximum allowed true physical deviation per month
Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], 25.0**2)    # Allowed true oil variance limit (~25 kbd)
np.fill_diagonal(Q[4:8, 4:8], 100.0**2)   # Allowed true gas variance limit (~100 mmcfd)
np.fill_diagonal(Q[8:12, 8:12], 0.02**2)  # Allowed monthly GOR drift

# Age definitions mapped from current evaluation boundary (June 2026)
reporting_rates_by_age = {
    0: np.array([1e-5]*8), # June 2026 (Age 0)
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]), # May 2026 (Age 1)
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]), # Apr 2026 (Age 2)
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]), # Mar 2026 (Age 3)
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]), # Feb 2026 (Age 4)
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])  # Mature History (Age 5+)
}

well_std_by_age = {0: 1e6, 1: 300, 2: 150, 3: 80, 4: 40, 5: 15}
eia_std_by_age  = {0: 1e-4, 1: 1e-4, 2: 1e-4, 3: 1e-4, 4: 1e-4, 5: 1e-4} # Strict commodity check
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1e6, 5: 1e6}

# ==========================================
# 3. ADVANCED PRE-LOOP INITIAL CALIBRATION
# ==========================================
# We parse the first three rows to establish a clean, non-volatile market proportion seed
cal_rows = df.head(3)
cal_oil_shares = []
cal_gas_shares = []

for _, c_row in cal_rows.iterrows():
    raw_o = np.array([get_val(c_row, 'o', 'Permian'), get_val(c_row, 'o', 'EagleFord'), get_val(c_row, 'o', 'Haynesville'), get_val(c_row, 'o', 'Other')])
    raw_g = np.array([get_val(c_row, 'g', 'Permian'), get_val(c_row, 'g', 'EagleFord'), get_val(c_row, 'g', 'Haynesville'), get_val(c_row, 'g', 'Other')])
    if raw_o.sum() > 0: cal_oil_shares.append(raw_o / raw_o.sum())
    if raw_g.sum() > 0: cal_gas_shares.append(raw_g / raw_g.sum())

# Determine the calm baseline allocation
avg_oil_shares = np.mean(cal_oil_shares, axis=0) if cal_oil_shares else np.array([0.5, 0.2, 0.1, 0.2])
avg_gas_shares = np.mean(cal_gas_shares, axis=0) if cal_gas_shares else np.array([0.4, 0.2, 0.3, 0.1])
initial_gor = np.array([2.5, 3.0, 150.0, 4.0])

# Seed Day 1 using calibrated ratios scaled precisely against Day 1's actual state totals
first_row = df.iloc[0]
total_tx_oil = get_val(first_row, 'o', 'TX')
total_tx_gas = get_val(first_row, 'g', 'TX')

# Secure a robust fallback scale if headers are missing metrics on entry row
oil_anchor = total_tx_oil if total_tx_oil > 0 else 5000
gas_anchor = total_tx_gas if total_tx_gas > 0 else 15000

calibrated_initial_oil = avg_oil_shares * oil_anchor
calibrated_initial_gas = avg_gas_shares * gas_anchor

# Lock in our stable, calibrated start state vector x
x = np.concatenate([calibrated_initial_oil, calibrated_initial_gas, initial_gor])
P = np.eye(n_states) * 10**2 # Low variance entry point since it's calibrated

# ==========================================
# 4. RECURSION TIMELINE PROCESSING LOOP
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5)

    # --- MODEL STEP 1: TIME PREDICTION (Carry memory forward) ---
    if idx > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    # Extract observations
    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # --- PHASE A: FUSE BOTTOM-UP WELL DATA ---
    if age > 0 and well_raw.sum() > 0:
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # --- PHASE B: FUSE STATE EIA-914 DATA ---
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        y_e = eia_observed - np.dot(H_eia, x)
        S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
        K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
        x = x + np.dot(K_e, y_e)
        P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # --- PHASE C: EXTENDED GOR FILTER LAYER ---
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]
        H_gor[b, b+4] = 1.0
        H_gor[b, b+8] = -x[b]
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)
    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # --- PHASE D: ENFORCE BOUNDARY FLOOR PROTECTION ---
    # Oil floor validation
    for b in range(3):
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]
            x[3] = x[3] - discrepancy # Deduct discrepancy from 'Other'

    # Gas floor validation
    for b in range(4, 7):
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]
            x[7] = x[7] - discrepancy # Deduct discrepancy from 'Other'

    # Final protection for the remainder 'Other' vectors
    if x[3] < well_raw[3]: x[3] = well_raw[3]
    if x[7] < well_raw[7]: x[7] = well_raw[7]

    # Map output components cleanly to standard array columns
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 5. GENERATE FINAL DATA VIEW
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== FINAL RECONCILED PRODUCTION MODEL ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== FINAL RECONCILED PRODUCTION MODEL ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6267.0                   1532.8                       31.6                351.0                30280.5                  12391.5                    15527.0               7270.0                    2.7                      3.0                      150.0                  4.0
2026-02    4                 6662.0                   1420.8                       25.0                330.0                32046.1                  12515.5                    16284.0               7190.0                    2.7                      3.0                      150.0                  4.0
2026-03    3                 6608.0                   1

In [2]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Normalize column names: strip spaces, underscores, and convert to lowercase
# Example: "o_TX" -> "otx", "g_Permian" -> "gpermian"
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return row.get(clean_name, 0)

# Define our 12 internal hidden state variables
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# ========================================================
# 2. DYNAMICALLY CALCULATE GEOGRAPHIC ALLOCATION MATRIX (H_eia)
# ========================================================
# Use the first 12 months of clean data to calculate exact file proportions
historical_mature_sample = df.head(12)

avg_tx_oil_total = historical_mature_sample['otx'].mean() if 'otx' in df.columns else 1.0
avg_nm_oil_total = historical_mature_sample['onm'].mean() if 'onm' in df.columns else 0.0
avg_la_oil_total = historical_mature_sample['ola'].mean() if 'ola' in df.columns else 0.0

avg_tx_gas_total = historical_mature_sample['gtx'].mean() if 'gtx' in df.columns else 1.0
avg_nm_gas_total = historical_mature_sample['gnm'].mean() if 'gnm' in df.columns else 0.0
avg_la_gas_total = historical_mature_sample['gla'].mean() if 'gla' in df.columns else 0.0

avg_permian_oil = historical_mature_sample['opermian'].mean() if 'opermian' in df.columns else 1.0
avg_eagleford_oil = historical_mature_sample['oeagleford'].mean() if 'oeagleford' in df.columns else 0.0
avg_other_oil = historical_mature_sample['oother'].mean() if 'oother' in df.columns else 0.0

avg_permian_gas = historical_mature_sample['gpermian'].mean() if 'gpermian' in df.columns else 1.0
avg_eagleford_gas = historical_mature_sample['geagleford'].mean() if 'geagleford' in df.columns else 0.0
avg_haynesville_gas = historical_mature_sample['ghaynesville'].mean() if 'ghaynesville' in df.columns else 0.0
avg_other_gas = historical_mature_sample['gother'].mean() if 'gother' in df.columns else 0.0

# Calculate exact physical allocation weights based on YOUR file averages
permian_in_tx_oil_ratio = np.clip((avg_tx_oil_total - avg_eagleford_oil) / max(avg_permian_oil, 1), 0.50, 0.80)
permian_in_nm_oil_ratio = 1.0 - permian_in_tx_oil_ratio

other_in_tx_oil_ratio = np.clip((avg_tx_oil_total - avg_eagleford_oil - (permian_in_tx_oil_ratio * avg_permian_oil)) / max(avg_other_oil, 1), 0.0, 1.0)
other_in_nm_oil_ratio = np.clip((avg_nm_oil_total - (permian_in_nm_oil_ratio * avg_permian_oil)) / max(avg_other_oil, 1), 0.0, 1.0)
other_in_la_oil_ratio = 1.0 - other_in_tx_oil_ratio - other_in_nm_oil_ratio

permian_in_tx_gas_ratio = np.clip((avg_tx_gas_total - avg_eagleford_gas) / max(avg_permian_gas, 1), 0.50, 0.80)
permian_in_nm_gas_ratio = 1.0 - permian_in_tx_gas_ratio

other_in_tx_gas_ratio = np.clip((avg_tx_gas_total - avg_eagleford_gas - (permian_in_tx_gas_ratio * avg_permian_gas)) / max(avg_other_gas, 1), 0.0, 1.0)
other_in_nm_gas_ratio = np.clip((avg_nm_gas_total - (permian_in_nm_gas_ratio * avg_permian_gas)) / max(avg_other_gas, 1), 0.0, 1.0)
other_in_la_gas_ratio = 1.0 - other_in_tx_gas_ratio - other_in_nm_gas_ratio

# Populate H_eia using your unique, file-calculated allocation parameters
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [permian_in_tx_oil_ratio, 1.0, 0.0, other_in_tx_oil_ratio]   # TX Total Oil
H_eia[1, 0:4] = [permian_in_nm_oil_ratio, 0.0, 0.0, other_in_nm_oil_ratio]   # NM Total Oil
H_eia[2, 0:4] = [0.0,                     0.0, 1.0, other_in_la_oil_ratio]   # LA Total Oil
H_eia[3, 4:8] = [permian_in_tx_gas_ratio, 1.0, 0.0, other_in_tx_gas_ratio]   # TX Total Gas
H_eia[4, 4:8] = [permian_in_nm_gas_ratio, 0.0, 0.0, other_in_nm_gas_ratio]   # NM Total Gas
H_eia[5, 4:8] = [0.0,                     0.0, 1.0, other_in_la_gas_ratio]   # LA Total Gas

# ==========================================
# 3. RUN PARAMETERS BY DATA AGE
# ==========================================
F = np.eye(n_states)

Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], 25.0**2)    # Max monthly structural oil shift
np.fill_diagonal(Q[4:8, 4:8], 100.0**2)   # Max monthly structural gas shift
np.fill_diagonal(Q[8:12, 8:12], 0.02**2)  # Max GOR geological drift

reporting_rates_by_age = {
    0: np.array([1e-5]*8),
    1: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50]),
    2: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]),
    3: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]),
    4: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]),
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95])
}

well_std_by_age = {0: 1e6, 1: 300, 2: 150, 3: 80, 4: 40, 5: 15}
eia_std_by_age  = {0: 1e-4, 1: 1e-4, 2: 1e-4, 3: 1e-4, 4: 1e-4, 5: 1e-4} # Strict check
gor_constraint_std_by_age = {0: 0.1, 1: 0.1, 2: 0.2, 3: 5.0, 4: 1e6, 5: 1e6}

# ==========================================
# 4. INITIALIZE CALIBRATED SEED
# ==========================================
first_row = df.iloc[0]
avg_oil_shares = np.array([avg_permian_oil, avg_eagleford_oil, 0.0, avg_other_oil]) / (avg_permian_oil + avg_eagleford_oil + avg_other_oil)
avg_gas_shares = np.array([avg_permian_gas, avg_eagleford_gas, avg_haynesville_gas, avg_other_gas]) / (avg_permian_gas + avg_eagleford_gas + avg_haynesville_gas + avg_other_gas)
initial_gor = np.array([2.5, 3.0, 150.0, 4.0])

total_tx_oil = get_val(first_row, 'o', 'TX')
total_tx_gas = get_val(first_row, 'g', 'TX')
oil_anchor = total_tx_oil if total_tx_oil > 0 else 5000
gas_anchor = total_tx_gas if total_tx_gas > 0 else 15000

x = np.concatenate([avg_oil_shares * oil_anchor, avg_gas_shares * gas_anchor, initial_gor])
P = np.eye(n_states) * 10**2

# ==========================================
# 5. RECURSIVE TIMELINE PROCESS LOOP
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]
    age = (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month)
    lookup_age = min(age, 5)

    # --- MODEL STEP 1: TIME PREDICTION ---
    if idx > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, 0)

    # --- PHASE A: BOTTOM-UP WELL FUSION ---
    if age > 0 and well_raw.sum() > 0:
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)

        y_w = well_scaled - np.dot(H_w, x)
        S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
        K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
        x = x + np.dot(K_w, y_w)
        P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # --- PHASE B: TOP-DOWN EIA STATE FUSION ---
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        y_e = eia_observed - np.dot(H_eia, x)
        S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
        K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
        x = x + np.dot(K_e, y_e)
        P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # --- PHASE C: EXTENDED GOR FILTER ---
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]
        H_gor[b, b+4] = 1.0
        H_gor[b, b+8] = -x[b]
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)
    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    # --- PHASE D: ENFORCE BOUNDARY FLOOR PROTECTION ---
    for b in range(3):
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]
            x[3] = x[3] - discrepancy

    for b in range(4, 7):
        if x[b] < well_raw[b]:
            discrepancy = well_raw[b] - x[b]
            x[b] = well_raw[b]
            x[7] = x[7] - discrepancy

    if x[3] < well_raw[3]: x[3] = well_raw[3]
    if x[7] < well_raw[7]: x[7] = well_raw[7]

    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 6. GENERATE FINAL MATRIX DATA VIEW
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== AUTOMATED DYNAMIC ALLOCATION OUTCOMES ===")
print(final_results_df.tail(6).round(1).to_string(index=False))


=== AUTOMATED DYNAMIC ALLOCATION OUTCOMES ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6429.0                   1107.0                       22.0                351.0                28705.3                   8133.0                    16508.9               7270.0                    2.7                      3.0                      150.0                  4.0
2026-02    4                 6690.3                   1085.0                       25.0                382.7                30467.7                   8122.0                    17258.5               7190.0                    2.7                      3.0                      150.0                  4.0
2026-03    3                 8020.3                

In [3]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return float(row.get(clean_name, 0.0))

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# ========================================================
# 2. CALIBRATE GEOGRAPHIC BASIN-TO-STATE MATRIX (H_eia)
# ========================================================
# Benchmark geographical allocation:
# - Permian: TX (~65-75%) and NM (~25-35%)
# - Eagle Ford: 100% TX
# - Haynesville: LA (~65-75%) and TX (~25-35%)
# - Other: Distributed across TX, NM, LA

historical_sample = df.head(12)

# Compute representative averages if present in well-level columns
avg_p_oil = historical_sample['opermian'].mean() if 'opermian' in df.columns else 4500.0
avg_ef_oil = historical_sample['oeagleford'].mean() if 'oeagleford' in df.columns else 1100.0
avg_hv_oil = historical_sample['ohaynesville'].mean() if 'ohaynesville' in df.columns else 35.0
avg_oth_oil = historical_sample['oother'].mean() if 'oother' in df.columns else 400.0

avg_p_gas = historical_sample['gpermian'].mean() if 'gpermian' in df.columns else 18000.0
avg_ef_gas = historical_sample['geagleford'].mean() if 'geagleford' in df.columns else 5500.0
avg_hv_gas = historical_sample['ghaynesville'].mean() if 'ghaynesville' in df.columns else 12500.0
avg_oth_gas = historical_sample['gother'].mean() if 'gother' in df.columns else 3000.0

# Typical geographic splits (empirically derived from EIA / state production)
permian_oil_tx_share = 0.70
permian_oil_nm_share = 1.0 - permian_oil_tx_share

permian_gas_tx_share = 0.68
permian_gas_nm_share = 1.0 - permian_gas_tx_share

haynesville_gas_la_share = 0.72
haynesville_gas_tx_share = 1.0 - haynesville_gas_la_share

haynesville_oil_la_share = 0.80
haynesville_oil_tx_share = 1.0 - haynesville_oil_la_share

# Proportional distribution of "Other" (TX, NM, LA)
other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

# Build H_eia (6 constraints: TX_oil, NM_oil, LA_oil, TX_gas, NM_gas, LA_gas)
H_eia = np.zeros((6, n_states))

# Oil rows
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]  # TX Oil
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]  # NM Oil
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]  # LA Oil

# Gas rows
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]  # TX Gas
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]  # NM Gas
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]  # LA Gas

# ==========================================
# 3. RUN PARAMETERS BY DATA AGE
# ==========================================
F = np.eye(n_states)

Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], 30.0**2)   # Oil process noise
np.fill_diagonal(Q[4:8, 4:8], 80.0**2)   # Gas process noise
np.fill_diagonal(Q[8:12, 8:12], 0.05**2) # GOR drift

reporting_rates_by_age = {
    0: np.array([1e-3]*8),
    1: np.array([0.55, 0.52, 0.60, 0.50, 0.55, 0.52, 0.50, 0.50]),
    2: np.array([0.72, 0.70, 0.75, 0.68, 0.72, 0.70, 0.65, 0.65]),
    3: np.array([0.86, 0.84, 0.88, 0.82, 0.85, 0.83, 0.80, 0.80]),
    4: np.array([0.92, 0.90, 0.94, 0.88, 0.92, 0.90, 0.89, 0.87]),
    5: np.array([0.97, 0.96, 0.98, 0.95, 0.97, 0.96, 0.96, 0.95])
}

well_std_by_age = {0: 1e5, 1: 250, 2: 120, 3: 60, 4: 25, 5: 10}
eia_std_by_age  = {0: 5.0, 1: 5.0, 2: 5.0, 3: 5.0, 4: 5.0, 5: 5.0}  # Balanced weight
gor_constraint_std_by_age = {0: 0.5, 1: 0.5, 2: 1.0, 3: 5.0, 4: 50.0, 5: 100.0}

# ==========================================
# 4. INITIALIZE CALIBRATED SEED
# ==========================================
first_row = df.iloc[0]

# Total system anchors across TX, NM, and LA
total_initial_oil = get_val(first_row, 'o', 'TX') + get_val(first_row, 'o', 'NM') + get_val(first_row, 'o', 'LA')
total_initial_gas = get_val(first_row, 'g', 'TX') + get_val(first_row, 'g', 'NM') + get_val(first_row, 'g', 'LA')

total_initial_oil = total_initial_oil if total_initial_oil > 0 else 6000.0
total_initial_gas = total_initial_gas if total_initial_gas > 0 else 35000.0

oil_seed_shares = np.array([avg_p_oil, avg_ef_oil, avg_hv_oil, avg_oth_oil])
oil_seed_shares = oil_seed_shares / oil_seed_shares.sum()

gas_seed_shares = np.array([avg_p_gas, avg_ef_gas, avg_hv_gas, avg_oth_gas])
gas_seed_shares = gas_seed_shares / gas_seed_shares.sum()

initial_gor = np.array([
    avg_p_gas / max(avg_p_oil, 1),
    avg_ef_gas / max(avg_ef_oil, 1),
    avg_hv_gas / max(avg_hv_oil, 1),
    avg_oth_gas / max(avg_oth_oil, 1)
])

x = np.concatenate([oil_seed_shares * total_initial_oil, gas_seed_shares * total_initial_gas, initial_gor])
P = np.eye(n_states) * 100.0

def kf_update(x_est, p_cov, z, H, R):
    """Numerically robust Kalman update (Joseph form)."""
    y = z - np.dot(H, x_est)
    S = np.dot(H, np.dot(p_cov, H.T)) + R
    K = np.dot(p_cov, np.dot(H.T, np.linalg.pinv(S)))
    x_new = x_est + np.dot(K, y)
    I_KH = np.eye(len(x_est)) - np.dot(K, H)
    p_new = np.dot(I_KH, np.dot(p_cov, I_KH.T)) + np.dot(K, np.dot(R, K.T))
    return x_new, p_new

# ==========================================
# 5. RECURSIVE TIMELINE PROCESS LOOP
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    lookup_age = min(age, 5)

    # Time prediction
    if idx > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    # Phase A: Bottom-up Well Fusion
    if age > 0 and well_raw.sum() > 0:
        rates = reporting_rates_by_age[lookup_age]
        well_scaled = np.where(well_raw > 0, well_raw / rates, x[0:8])
        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)
        x, P = kf_update(x, P, well_scaled, H_w, R_w)

    # Phase B: Top-down State Totals Fusion
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        x, P = kf_update(x, P, eia_observed, H_eia, R_e)

    # Phase C: Extended GOR Consistency Filter
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b] = -x[b+8]     # -GOR * d(Oil)
        H_gor[b, b+4] = 1.0       # +1 * d(Gas)
        H_gor[b, b+8] = -x[b]     # -Oil * d(GOR)

    gor_residuals = np.zeros(4)
    gor_observed = x[4:8] - (x[0:4] * x[8:12])
    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)
    x, P = kf_update(x, P, np.zeros(4), H_gor, R_gor)

    # Phase D: Enforce Physical Non-negativity & Well-level minimum floors
    for b in range(8):
        if well_raw[b] > 0 and x[b] < well_raw[b]:
            x[b] = well_raw[b]
        elif x[b] < 0:
            x[b] = 0.0

    # Keep GOR within physical boundaries
    x[8:12] = np.clip(x[8:12], 0.1, 500.0)

    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 6. OUTPUT RESULTS
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== CORRECTED DYNAMIC ALLOCATION OUTCOMES ===")
print(final_results_df.tail(6).round(1).to_string(index=False))

=== CORRECTED DYNAMIC ALLOCATION OUTCOMES ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6504.2                   1107.0                       22.0                351.0                31276.3                   8514.1                    15527.0               7270.0                    2.4                      4.8                      447.8                 14.0
2026-02    4                 7139.4                   1085.0                       25.0                330.0                35547.1                   8122.0                    16284.0               7190.0                    2.5                      5.4                      444.5                 14.2
2026-03    3                 7190.0                

In [4]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return float(row.get(clean_name, 0.0))

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)

# ==========================================
# 2. CALIBRATE ALLOCATION MATRIX (H_eia)
# ==========================================
historical_sample = df.head(12)

# Geographic allocation splits
permian_oil_tx_share, permian_oil_nm_share = 0.70, 0.30
permian_gas_tx_share, permian_gas_nm_share = 0.68, 0.32

haynesville_gas_la_share, haynesville_gas_tx_share = 0.70, 0.30
haynesville_oil_la_share, haynesville_oil_tx_share = 0.80, 0.20

other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

H_eia = np.zeros((6, n_states))
# TX, NM, LA Oil
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]

# TX, NM, LA Gas
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]

# ==========================================
# 3. RUN PARAMETERS & NOISE SCHEDULES
# ==========================================
F = np.eye(n_states)

# Reporting completeness factors by vintage age
reporting_rates_by_age = {
    0: np.array([0.15, 0.15, 0.20, 0.15, 0.15, 0.15, 0.20, 0.15]),
    1: np.array([0.65, 0.60, 0.70, 0.60, 0.65, 0.60, 0.68, 0.60]),
    2: np.array([0.85, 0.82, 0.88, 0.80, 0.85, 0.82, 0.85, 0.80]),
    3: np.array([0.94, 0.92, 0.95, 0.90, 0.94, 0.92, 0.93, 0.90]),
    4: np.array([0.98, 0.97, 0.98, 0.95, 0.98, 0.97, 0.97, 0.95]),
    5: np.array([1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00])
}

# Standard deviations scaled realistically
# (Older data is trusted more; younger data has higher uncertainty)
well_std_by_age = {0: 1e6, 1: 500, 2: 250, 3: 100, 4: 40, 5: 15}
eia_std_by_age  = {0: 50.0, 1: 30.0, 2: 20.0, 3: 10.0, 4: 5.0, 5: 5.0}

# FIX: GOR constraint noise should NOT be overly tight on recent vintages
gor_constraint_std_by_age = {0: 500.0, 1: 250.0, 2: 100.0, 3: 50.0, 4: 20.0, 5: 10.0}

# ==========================================
# 4. INITIALIZE CALIBRATED SEED
# ==========================================
first_row = df.iloc[0]

total_initial_oil = get_val(first_row, 'o', 'TX') + get_val(first_row, 'o', 'NM') + get_val(first_row, 'o', 'LA')
total_initial_gas = get_val(first_row, 'g', 'TX') + get_val(first_row, 'g', 'NM') + get_val(first_row, 'g', 'LA')

total_initial_oil = total_initial_oil if total_initial_oil > 0 else 6000.0
total_initial_gas = total_initial_gas if total_initial_gas > 0 else 35000.0

# Initial distribution seed shares
oil_seed_shares = np.array([0.72, 0.18, 0.01, 0.09])
gas_seed_shares = np.array([0.55, 0.16, 0.22, 0.07])

initial_gor = (gas_seed_shares * total_initial_gas) / np.maximum(oil_seed_shares * total_initial_oil, 1.0)

x = np.concatenate([oil_seed_shares * total_initial_oil, gas_seed_shares * total_initial_gas, initial_gor])
P = np.eye(n_states) * 100.0

def kf_update(x_est, p_cov, z, H, R):
    y = z - np.dot(H, x_est)
    S = np.dot(H, np.dot(p_cov, H.T)) + R
    K = np.dot(p_cov, np.dot(H.T, np.linalg.pinv(S)))
    x_new = x_est + np.dot(K, y)
    I_KH = np.eye(len(x_est)) - np.dot(K, H)
    p_new = np.dot(I_KH, np.dot(p_cov, I_KH.T)) + np.dot(K, np.dot(R, K.T))
    return x_new, p_new

# ==========================================
# 5. RECURSIVE TIMELINE PROCESS LOOP
# ==========================================
fused_output_records = []
current_evaluation_month = pd.to_datetime("2026-06-01")

for idx, row in df.iterrows():
    month_dt = row.iloc[0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    lookup_age = min(age, 5)

    # Dynamic Q: Scale max structural shift relative to each basin's current volume (max 3% per month)
    Q = np.zeros((n_states, n_states))
    for i in range(8):
        Q[i, i] = max(10.0, x[i] * 0.03)**2
    for i in range(8, 12):
        Q[i, i] = 0.05**2  # GOR process noise

    # Time prediction
    if idx > 0:
        x_prev = x.copy()
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q
    else:
        x_prev = x.copy()

    eia_observed = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    # Phase A: Bottom-up Well Fusion with Outlier / Completeness Floor
    if age > 0 and well_raw.sum() > 0:
        rates = reporting_rates_by_age[lookup_age]
        # Guard: If raw well volume implies a sudden >40% crash due to filing delays, blend with prior state
        scaled_candidates = np.where(well_raw > 0, well_raw / rates, x[0:8])
        well_scaled = np.maximum(scaled_candidates, x_prev[0:8] * 0.90)  # Max 10% month-over-month decline

        H_w = np.zeros((8, n_states))
        H_w[0:8, 0:8] = np.eye(8)
        R_w = np.eye(8) * (well_std_by_age[lookup_age]**2)
        x, P = kf_update(x, P, well_scaled, H_w, R_w)

    # Phase B: Top-down State EIA Fusion
    if eia_observed.sum() > 0:
        R_e = np.eye(6) * (eia_std_by_age[lookup_age]**2)
        x, P = kf_update(x, P, eia_observed, H_eia, R_e)

    # Phase C: GOR Consistency Filter (Stabilized)
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        H_gor[b, b]   = -x[b+8]     # -GOR
        H_gor[b, b+4] = 1.0         # +1 Gas
        H_gor[b, b+8] = -x[b]       # -Oil

    R_gor = np.eye(4) * (gor_constraint_std_by_age[lookup_age]**2)
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))
    x, P = kf_update(x, P, gor_residuals, H_gor, R_gor)

    # Phase D: Physical Bounds
    for b in range(8):
        x[b] = max(x[b], well_raw[b], 0.0)

    # Bound GORs to realistic geological ranges (Mcf/Bbl)
    # Permian: 1.5 - 6.0, Eagle Ford: 2.0 - 8.0, Haynesville: 50.0 - 500.0, Other: 1.0 - 15.0
    x[8]  = np.clip(x[8], 1.2, 8.0)
    x[9]  = np.clip(x[9], 1.5, 10.0)
    x[10] = np.clip(x[10], 40.0, 600.0)
    x[11] = np.clip(x[11], 1.0, 20.0)

    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        x[0], x[1], x[2], x[3],
        x[4], x[5], x[6], x[7],
        x[8], x[9], x[10], x[11]
    ])

# ==========================================
# 6. OUTPUT RESULTS
# ==========================================
output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== STABILIZED DYNAMIC ALLOCATION OUTCOMES ===")
print(final_results_df.tail(6).round(1).to_string(index=False))

=== STABILIZED DYNAMIC ALLOCATION OUTCOMES ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6365.3                   1107.0                      187.8                351.0                32478.4                   8133.0                    15527.0               7270.0                    1.2                      4.3                       91.4                 14.9
2026-02    4                 6662.0                   1311.0                      156.6                330.0                36314.4                   8122.0                    16284.0               7190.0                    7.7                      7.7                       94.7                 15.5
2026-03    3                 6608.0               

In [5]:
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return float(row.get(clean_name, 0.0))

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)
T = len(df)

# ==========================================
# 2. CALIBRATE ALLOCATION MATRIX (H_eia)
# ==========================================
# Fixed, geologically stable distribution shares
permian_oil_tx_share, permian_oil_nm_share = 0.70, 0.30
permian_gas_tx_share, permian_gas_nm_share = 0.68, 0.32

haynesville_gas_la_share, haynesville_gas_tx_share = 0.72, 0.28
haynesville_oil_la_share, haynesville_oil_tx_share = 0.80, 0.20

other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

H_eia = np.zeros((6, n_states))
# Oil rows (TX, NM, LA)
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]

# Gas rows (TX, NM, LA)
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]

# ==========================================
# 3. SMOOTH PROCESS NOISE & MEASUREMENT MODELS
# ==========================================
F = np.eye(n_states)

# Process noise: limits physical step change per month
# Prevents sudden unphysical jumps (std dev: Oil ~1.5%, Gas ~1.5%, GOR ~0.5%)
Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], [35.0**2, 15.0**2, 2.0**2, 10.0**2])    # Perm, EF, HV, Other (Oil kb/d)
np.fill_diagonal(Q[4:8, 4:8], [120.0**2, 40.0**2, 80.0**2, 25.0**2]) # Perm, EF, HV, Other (Gas MMcf/d)
np.fill_diagonal(Q[8:12, 8:12], [0.03**2, 0.04**2, 1.0**2, 0.05**2]) # GOR drift

# Reporting completeness factors
reporting_rates_by_age = {
    0: np.array([0.20, 0.20, 0.25, 0.20, 0.20, 0.20, 0.25, 0.20]),
    1: np.array([0.65, 0.62, 0.70, 0.60, 0.65, 0.62, 0.68, 0.60]),
    2: np.array([0.85, 0.83, 0.88, 0.82, 0.85, 0.83, 0.85, 0.82]),
    3: np.array([0.94, 0.93, 0.95, 0.92, 0.94, 0.93, 0.93, 0.92]),
    4: np.array([0.98, 0.97, 0.98, 0.96, 0.98, 0.97, 0.97, 0.96]),
    5: np.array([1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00])
}

# Measurement standard deviations (higher standard deviation = smoother fit)
well_std_by_age = {0: 1e5, 1: 300, 2: 150, 3: 80, 4: 40, 5: 20}
eia_std_by_age  = {0: 15.0, 1: 15.0, 2: 15.0, 3: 15.0, 4: 15.0, 5: 15.0}

# ==========================================
# 4. INITIALIZE CALIBRATED SEED
# ==========================================
first_row = df.iloc[0]

total_initial_oil = get_val(first_row, 'o', 'TX') + get_val(first_row, 'o', 'NM') + get_val(first_row, 'o', 'LA')
total_initial_gas = get_val(first_row, 'g', 'TX') + get_val(first_row, 'g', 'NM') + get_val(first_row, 'g', 'LA')
total_initial_oil = total_initial_oil if total_initial_oil > 0 else 6000.0
total_initial_gas = total_initial_gas if total_initial_gas > 0 else 35000.0

oil_seed = np.array([0.72, 0.18, 0.01, 0.09]) * total_initial_oil
gas_seed = np.array([0.55, 0.16, 0.22, 0.07]) * total_initial_gas
gor_seed = np.array([3.0, 3.8, 180.0, 4.5])

x0 = np.concatenate([oil_seed, gas_seed, gor_seed])
P0 = np.eye(n_states) * 100.0

# ==========================================
# 5. FORWARD KALMAN FILTER PASS
# ==========================================
x_pred = np.zeros((T, n_states))
P_pred = np.zeros((T, n_states, n_states))
x_filt = np.zeros((T, n_states))
P_filt = np.zeros((T, n_states, n_states))

current_evaluation_month = pd.to_datetime("2026-06-01")
x = x0.copy()
P = P0.copy()

for t in range(T):
    row = df.iloc[t]
    month_dt = row.iloc[0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    lookup_age = min(age, 5)

    # 1. Prediction step
    if t > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    x_pred[t] = x.copy()
    P_pred[t] = P.copy()

    # 2. Assemble Combined Observation Vector (Well + EIA)
    eia_obs = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])
    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    # Scaled bottom-up inputs
    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, x[0:8])

    # Joint observation matrices
    # Top 8 rows: Well estimates, Bottom 6 rows: EIA totals
    H_joint = np.zeros((14, n_states))
    H_joint[0:8, 0:8] = np.eye(8)
    H_joint[8:14, :] = H_eia

    z_joint = np.concatenate([well_scaled, eia_obs])

    R_joint = np.zeros((14, 14))
    np.fill_diagonal(R_joint[0:8, 0:8], well_std_by_age[lookup_age]**2)
    np.fill_diagonal(R_joint[8:14, 8:14], eia_std_by_age[lookup_age]**2)

    # 3. Single Unified Update (Joseph Form)
    y = z_joint - np.dot(H_joint, x)
    S = np.dot(H_joint, np.dot(P, H_joint.T)) + R_joint
    K = np.dot(P, np.dot(H_joint.T, np.linalg.pinv(S)))

    x = x + np.dot(K, y)
    I_KH = np.eye(n_states) - np.dot(K, H_joint)
    P = np.dot(I_KH, np.dot(P, I_KH.T)) + np.dot(K, np.dot(R_joint, K.T))

    # Update GOR state passively to reflect smooth trend
    for b in range(4):
        if x[b] > 1.0:
            instant_gor = max(0.1, x[b+4] / x[b])
            x[b+8] = 0.90 * x[b+8] + 0.10 * instant_gor  # Exponential moving average filter

    x_filt[t] = x.copy()
    P_filt[t] = P.copy()

# ==========================================
# 6. BACKWARD RAUCH-TUNG-STRIEBEL (RTS) SMOOTHER
# ==========================================
# This eliminates high-frequency noise and retroactively stabilizes recent months
x_smooth = np.zeros_like(x_filt)
P_smooth = np.zeros_like(P_filt)
x_smooth[-1] = x_filt[-1]
P_smooth[-1] = P_filt[-1]

for t in range(T - 2, -1, -1):
    C = np.dot(P_filt[t], np.dot(F.T, np.linalg.pinv(P_pred[t+1])))
    x_smooth[t] = x_filt[t] + np.dot(C, (x_smooth[t+1] - x_pred[t+1]))
    P_smooth[t] = P_filt[t] + np.dot(C, np.dot(P_smooth[t+1] - P_pred[t+1], C.T))

# ==========================================
# 7. GENERATE FINAL MATRIX DATA VIEW
# ==========================================
fused_output_records = []
for t in range(T):
    month_dt = df.iloc[t, 0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        *x_smooth[t]
    ])

output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== SMOOTHED & BALANCED DYNAMIC ALLOCATION ===")
print(final_results_df.tail(6).round(1).to_string(index=False))

=== SMOOTHED & BALANCED DYNAMIC ALLOCATION ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6481.8                    896.1                       15.9                257.4                30123.7                   8648.7                    13664.2               6706.8                    4.7                      9.5                      916.4                 26.1
2026-02    4                 6930.7                    795.6                       16.5                232.9                33328.4                   8187.0                    13307.7               6400.6                    4.7                      9.5                      916.9                 26.1
2026-03    3                 7176.8               

In [6]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return float(row.get(clean_name, 0.0))

state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)
T = len(df)

# ==========================================
# 2. CALIBRATE ALLOCATION MATRIX (H_eia)
# ==========================================
permian_oil_tx_share, permian_oil_nm_share = 0.70, 0.30
permian_gas_tx_share, permian_gas_nm_share = 0.68, 0.32

haynesville_gas_la_share, haynesville_gas_tx_share = 0.72, 0.28
haynesville_oil_la_share, haynesville_oil_tx_share = 0.80, 0.20

other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

H_eia = np.zeros((6, n_states))
# Oil rows (TX, NM, LA)
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]

# Gas rows (TX, NM, LA)
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]

# ==========================================
# 3. SMOOTH PROCESS NOISE & MEASUREMENT MODELS
# ==========================================
F = np.eye(n_states)

Q = np.zeros((n_states, n_states))
np.fill_diagonal(Q[0:4, 0:4], [35.0**2, 15.0**2, 2.0**2, 10.0**2])
np.fill_diagonal(Q[4:8, 4:8], [120.0**2, 40.0**2, 80.0**2, 25.0**2])
np.fill_diagonal(Q[8:12, 8:12], [0.03**2, 0.04**2, 1.0**2, 0.05**2])

reporting_rates_by_age = {
    0: np.array([0.20, 0.20, 0.25, 0.20, 0.20, 0.20, 0.25, 0.20]),
    1: np.array([0.65, 0.62, 0.70, 0.60, 0.65, 0.62, 0.68, 0.60]),
    2: np.array([0.85, 0.83, 0.88, 0.82, 0.85, 0.83, 0.85, 0.82]),
    3: np.array([0.94, 0.93, 0.95, 0.92, 0.94, 0.93, 0.93, 0.92]),
    4: np.array([0.98, 0.97, 0.98, 0.96, 0.98, 0.97, 0.97, 0.96]),
    5: np.array([1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00])
}

well_std_by_age = {0: 1e5, 1: 300, 2: 150, 3: 80, 4: 40, 5: 20}
eia_std_by_age  = {0: 15.0, 1: 15.0, 2: 15.0, 3: 15.0, 4: 15.0, 5: 15.0}

# ==========================================
# 4. INITIALIZE CALIBRATED SEED
# ==========================================
first_row = df.iloc[0]

total_initial_oil = get_val(first_row, 'o', 'TX') + get_val(first_row, 'o', 'NM') + get_val(first_row, 'o', 'LA')
total_initial_gas = get_val(first_row, 'g', 'TX') + get_val(first_row, 'g', 'NM') + get_val(first_row, 'g', 'LA')
total_initial_oil = total_initial_oil if total_initial_oil > 0 else 6000.0
total_initial_gas = total_initial_gas if total_initial_gas > 0 else 35000.0

oil_seed = np.array([0.72, 0.18, 0.01, 0.09]) * total_initial_oil
gas_seed = np.array([0.55, 0.16, 0.22, 0.07]) * total_initial_gas
gor_seed = np.array([3.0, 3.8, 180.0, 4.5])

x0 = np.concatenate([oil_seed, gas_seed, gor_seed])
P0 = np.eye(n_states) * 100.0

# ==========================================
# 5. FORWARD KALMAN FILTER PASS
# ==========================================
x_pred = np.zeros((T, n_states))
P_pred = np.zeros((T, n_states, n_states))
x_filt = np.zeros((T, n_states))
P_filt = np.zeros((T, n_states, n_states))

current_evaluation_month = pd.to_datetime("2026-06-01")
x = x0.copy()
P = P0.copy()

for t in range(T):
    row = df.iloc[t]
    month_dt = row.iloc[0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    lookup_age = min(age, 5)

    if t > 0:
        x = np.dot(F, x)
        P = np.dot(F, np.dot(P, F.T)) + Q

    x_pred[t] = x.copy()
    P_pred[t] = P.copy()

    eia_obs = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])
    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    rates = reporting_rates_by_age[lookup_age]
    well_scaled = np.where(well_raw > 0, well_raw / rates, x[0:8])

    H_joint = np.zeros((14, n_states))
    H_joint[0:8, 0:8] = np.eye(8)
    H_joint[8:14, :] = H_eia
    z_joint = np.concatenate([well_scaled, eia_obs])

    R_joint = np.zeros((14, 14))
    np.fill_diagonal(R_joint[0:8, 0:8], well_std_by_age[lookup_age]**2)
    np.fill_diagonal(R_joint[8:14, 8:14], eia_std_by_age[lookup_age]**2)

    y = z_joint - np.dot(H_joint, x)
    S = np.dot(H_joint, np.dot(P, H_joint.T)) + R_joint
    K = np.dot(P, np.dot(H_joint.T, np.linalg.pinv(S)))

    x = x + np.dot(K, y)
    I_KH = np.eye(n_states) - np.dot(K, H_joint)
    P = np.dot(I_KH, np.dot(P, I_KH.T)) + np.dot(K, np.dot(R_joint, K.T))

    for b in range(4):
        if x[b] > 1.0:
            instant_gor = max(0.1, x[b+4] / x[b])
            x[b+8] = 0.90 * x[b+8] + 0.10 * instant_gor

    x_filt[t] = x.copy()
    P_filt[t] = P.copy()

# ==========================================
# 6. BACKWARD RTS SMOOTHER
# ==========================================
x_smooth = np.zeros_like(x_filt)
P_smooth = np.zeros_like(P_filt)
x_smooth[-1] = x_filt[-1]
P_smooth[-1] = P_filt[-1]

for t in range(T - 2, -1, -1):
    C = np.dot(P_filt[t], np.dot(F.T, np.linalg.pinv(P_pred[t+1])))
    x_smooth[t] = x_filt[t] + np.dot(C, (x_smooth[t+1] - x_pred[t+1]))
    P_smooth[t] = P_filt[t] + np.dot(C, np.dot(P_smooth[t+1] - P_pred[t+1], C.T))

# ==========================================
# 7. QUADRATIC CONSTRAINT PROJECTION (FLOORS >= REPORTED)
# ==========================================
# Smoothly project smoothed trajectories to strictly satisfy:
# (1) Estimated >= Reported Raw Well Data
# (2) Total EIA State Allocation Consistency
final_estimates = np.zeros_like(x_smooth)

for t in range(T):
    row = df.iloc[t]
    x_target = x_smooth[t].copy()

    well_raw = np.array([
        get_val(row, 'o', 'Permian'), get_val(row, 'o', 'EagleFord'), get_val(row, 'o', 'Haynesville'), get_val(row, 'o', 'Other'),
        get_val(row, 'g', 'Permian'), get_val(row, 'g', 'EagleFord'), get_val(row, 'g', 'Haynesville'), get_val(row, 'g', 'Other')
    ])

    eia_obs = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    # Lower bounds: at least reported well raw data for volumes, physical bounds for GOR
    lower_bounds = np.concatenate([
        np.maximum(0.0, well_raw),  # First 8 elements (Oil + Gas) >= well_raw
        [1.0, 1.5, 40.0, 1.0]       # GOR lower bounds
    ])

    upper_bounds = np.concatenate([
        [np.inf]*8,
        [10.0, 12.0, 600.0, 25.0]   # GOR upper bounds
    ])
    bounds = list(zip(lower_bounds, upper_bounds))

    # Minimize distance from the RTS-smoothed trajectory while strictly respecting bounds
    def objective(x_opt):
        # Weights: allow smooth small adjustments without distorting proportions
        weights = np.array([1.0, 1.0, 1.0, 1.0, 0.2, 0.2, 0.2, 0.2, 10.0, 10.0, 1.0, 10.0])
        return 0.5 * np.sum(weights * ((x_opt - x_target) ** 2))

    # Soft constraint to match EIA state totals if available
    constraints = []
    if eia_obs.sum() > 0:
        constraints.append({
            'type': 'eq',
            'fun': lambda x_opt: np.dot(H_eia, x_opt) - eia_obs
        })

    res = minimize(
        objective,
        x0=np.maximum(x_target, lower_bounds),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 200, 'ftol': 1e-4}
    )

    if res.success:
        final_estimates[t] = res.x
    else:
        # Fallback: strict floor enforcement
        fallback_x = x_target.copy()
        fallback_x[0:8] = np.maximum(fallback_x[0:8], well_raw)
        final_estimates[t] = fallback_x

# ==========================================
# 8. GENERATE FINAL MATRIX DATA VIEW
# ==========================================
fused_output_records = []
for t in range(T):
    month_dt = df.iloc[t, 0]
    age = max(0, (current_evaluation_month.year - month_dt.year) * 12 + (current_evaluation_month.month - month_dt.month))
    fused_output_records.append([
        month_dt.strftime('%Y-%m'), age,
        *final_estimates[t]
    ])

output_cols = ['Month', 'Age'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(fused_output_records, columns=output_cols)

print("=== FINAL CONSTRAINED ALLOCATION (ESTIMATED >= REPORTED) ===")
print(final_results_df.tail(6).round(1).to_string(index=False))

=== FINAL CONSTRAINED ALLOCATION (ESTIMATED >= REPORTED) ===
  Month  Age  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01    5                 6481.8                   1107.0                       22.0                351.0                30123.7                   8648.7                    15527.0               7270.0                    4.7                      9.5                      916.4                 26.1
2026-02    4                 6930.7                   1085.0                       25.0                330.0                33328.4                   8187.0                    16284.0               7190.0                    4.7                      9.5                      916.9                 26.1
2026-03    3                 7176.8 

In [8]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Standardize column names (lowercase, no underscores/spaces)
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    return float(row.get(clean_name, 0.0))

regions = ['Permian', 'EagleFord', 'Haynesville', 'Other']
state_elements = (
    [f'{r}_Oil' for r in regions] +
    [f'{r}_Gas' for r in regions] +
    [f'{r}_GOR' for r in regions]
)
n_states = len(state_elements)
T = len(df)

# ==========================================
# 2. STATE ALLOCATION GEOGRAPHY MATRIX (H_eia)
# ==========================================
# Regional distribution weights across TX, NM, and LA
permian_oil_tx_share, permian_oil_nm_share = 0.70, 0.30
permian_gas_tx_share, permian_gas_nm_share = 0.68, 0.32

haynesville_gas_la_share, haynesville_gas_tx_share = 0.72, 0.28
haynesville_oil_la_share, haynesville_oil_tx_share = 0.80, 0.20

other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

H_eia = np.zeros((6, 8))
# Oil rows (TX, NM, LA)
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]

# Gas rows (TX, NM, LA)
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]

# ==========================================
# 3. CONSTRAINED PROBABLE ESTIMATION ENGINE
# ==========================================
optimized_results = []

for t in range(T):
    row = df.iloc[t]
    month_dt = row.iloc[0]

    # --- A. Read Inputs ---
    # 1. Reported raw actuals (Hard lower floors)
    rep_oil = np.array([get_val(row, 'o', r) for r in regions])
    rep_gas = np.array([get_val(row, 'g', r) for r in regions])
    rep_vector = np.concatenate([rep_oil, rep_gas])

    # 2. Your Model Estimates (Priors / Targets)
    mod_oil = np.array([get_val(row, 'mo', r) for r in regions])
    mod_gas = np.array([get_val(row, 'mg', r) for r in regions])
    mod_vector = np.concatenate([mod_oil, mod_gas])

    # Fallback if model estimate column is empty: use prior row or non-zero floor
    if mod_vector.sum() == 0:
        if t > 0:
            mod_vector = optimized_results[-1][2:10].copy()
        else:
            mod_vector = np.maximum(rep_vector, 1.0)

    # 3. EIA State Totals (TX, NM, LA)
    eia_obs = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ])

    # --- B. Define Strict Bounds & Target Targets ---
    lower_bounds = np.maximum(0.0, rep_vector)

    # Upper bound logic:
    # If Reported > Model Estimate, upper bound expands to allow Reported,
    # but stays tightly bounded so it doesn't overshoot.
    upper_bounds = np.zeros(8)
    for i in range(8):
        if rep_vector[i] > mod_vector[i]:
            upper_bounds[i] = rep_vector[i] * 1.05  # Tight corridor around reported
        else:
            upper_bounds[i] = max(mod_vector[i], lower_bounds[i])

    bounds = list(zip(lower_bounds, upper_bounds))

    # --- C. Initial Guess ---
    # Start at model estimates or clamp at lower floor
    x_init = np.clip(mod_vector, lower_bounds, upper_bounds)

    # --- D. Formulate Optimization Objective ---
    def objective(x):
        # 1. Fit to Prior / Model Target:
        # Distance from your model estimates (normalized by scale)
        scale = np.maximum(mod_vector, 10.0)
        dist_to_model = np.sum(((x - mod_vector) / scale) ** 2)

        # 2. Penalty if Reported > Model: Stay as close to Reported as possible
        overshoot_penalty = 0.0
        for i in range(8):
            if rep_vector[i] > mod_vector[i]:
                overshoot_penalty += 10.0 * ((x[i] - rep_vector[i]) / scale[i]) ** 2

        # 3. Smoothness prior with previous month
        smoothness_penalty = 0.0
        if t > 0:
            x_prev = optimized_results[-1][2:10]
            smoothness_penalty = 0.5 * np.sum(((x - x_prev) / scale) ** 2)

        # 4. EIA reconciliation soft term
        eia_penalty = 0.0
        if eia_obs.sum() > 0:
            eia_diff = np.dot(H_eia, x) - eia_obs
            eia_scale = np.maximum(eia_obs, 50.0)
            eia_penalty = 2.0 * np.sum((eia_diff / eia_scale) ** 2)

        return dist_to_model + overshoot_penalty + smoothness_penalty + eia_penalty

    # --- E. Solve ---
    res = minimize(
        objective,
        x0=x_init,
        method='SLSQP',
        bounds=bounds,
        options={'maxiter': 300, 'ftol': 1e-5}
    )

    if res.success:
        x_final = res.x
    else:
        # Failsafe: clamp to strictly satisfy lower/upper bounds
        x_final = np.clip(x_init, lower_bounds, upper_bounds)

    # Calculate GOR from final converged oil and gas volumes
    gor_final = np.zeros(4)
    for b in range(4):
        oil_vol = x_final[b]
        gas_vol = x_final[b+4]
        gor_final[b] = (gas_vol / oil_vol) if oil_vol > 0.1 else 0.0

    # Store full record
    row_record = [
        month_dt.strftime('%Y-%m'),
        t,
        *x_final,      # 4 Oil + 4 Gas
        *gor_final     # 4 GOR
    ]
    optimized_results.append(row_record)

# ==========================================
# 4. FORMAT FINAL DATAFRAME
# ==========================================
output_cols = ['Month', 'Index'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(optimized_results, columns=output_cols)

print("=== FINAL CONSTRAINED ALLOCATION WITH USER ESTIMATES ===")
print(final_results_df.tail(12).round(1).to_string(index=False))

# Optional: Save to CSV
final_results_df.to_csv('Allocated_Production_Results.csv', index=False)

=== FINAL CONSTRAINED ALLOCATION WITH USER ESTIMATES ===
  Month  Index  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2025-07     18                 6597.0                   1188.0                       24.0                341.0                29493.0                   8120.0                    15081.0               6784.0                    4.5                      6.8                      628.4                 19.9
2025-08     19                 6602.0                   1167.0                       24.0                359.0                29769.0                   8226.0                    15444.0               6899.0                    4.5                      7.0                      643.5                 19.2
2025-09     20                 664

In [9]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

# Standardize column names (lowercase, no underscores/spaces)
df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    val = row.get(clean_name, 0.0)
    # Handle NaN / None / null entries cleanly
    if pd.isna(val) or val is None:
        return 0.0
    try:
        return float(val)
    except (ValueError, TypeError):
        return 0.0

regions = ['Permian', 'EagleFord', 'Haynesville', 'Other']
state_elements = (
    [f'{r}_Oil' for r in regions] +
    [f'{r}_Gas' for r in regions] +
    [f'{r}_GOR' for r in regions]
)
n_states = len(state_elements)
T = len(df)

# ==========================================
# 2. STATE ALLOCATION GEOGRAPHY MATRIX (H_eia)
# ==========================================
permian_oil_tx_share, permian_oil_nm_share = 0.70, 0.30
permian_gas_tx_share, permian_gas_nm_share = 0.68, 0.32

haynesville_gas_la_share, haynesville_gas_tx_share = 0.72, 0.28
haynesville_oil_la_share, haynesville_oil_tx_share = 0.80, 0.20

other_oil_tx_share, other_oil_nm_share, other_oil_la_share = 0.75, 0.05, 0.20
other_gas_tx_share, other_gas_nm_share, other_gas_la_share = 0.60, 0.05, 0.35

H_eia = np.zeros((6, 8))
# Oil rows (TX, NM, LA)
H_eia[0, 0:4] = [permian_oil_tx_share, 1.0, haynesville_oil_tx_share, other_oil_tx_share]
H_eia[1, 0:4] = [permian_oil_nm_share, 0.0, 0.0,                      other_oil_nm_share]
H_eia[2, 0:4] = [0.0,                  0.0, haynesville_oil_la_share, other_oil_la_share]

# Gas rows (TX, NM, LA)
H_eia[3, 4:8] = [permian_gas_tx_share, 1.0, haynesville_gas_tx_share, other_gas_tx_share]
H_eia[4, 4:8] = [permian_gas_nm_share, 0.0, 0.0,                      other_gas_nm_share]
H_eia[5, 4:8] = [0.0,                  0.0, haynesville_gas_la_share, other_gas_la_share]

# ==========================================
# 3. ROBUST OPTIMIZATION ENGINE (NO-NAN)
# ==========================================
optimized_results = []

for t in range(T):
    row = df.iloc[t]
    month_dt = row.iloc[0]

    # --- A. Read Inputs with strict NaN handling ---
    rep_oil = np.array([get_val(row, 'o', r) for r in regions], dtype=float)
    rep_gas = np.array([get_val(row, 'g', r) for r in regions], dtype=float)
    rep_vector = np.nan_to_num(np.concatenate([rep_oil, rep_gas]), nan=0.0)

    mod_oil = np.array([get_val(row, 'mo', r) for r in regions], dtype=float)
    mod_gas = np.array([get_val(row, 'mg', r) for r in regions], dtype=float)
    mod_vector = np.nan_to_num(np.concatenate([mod_oil, mod_gas]), nan=0.0)

    # If model estimate is missing for the month, forward-fill from previous month
    if np.all(mod_vector <= 0.0):
        if t > 0:
            mod_vector = np.array(optimized_results[-1][2:10], dtype=float)
        else:
            mod_vector = np.maximum(rep_vector, 100.0)

    # For any individual zero element in mod_vector, inherit from prior or non-zero floor
    for i in range(8):
        if mod_vector[i] <= 0.0:
            if t > 0:
                mod_vector[i] = float(optimized_results[-1][2 + i])
            else:
                mod_vector[i] = max(float(rep_vector[i]), 10.0)

    eia_obs = np.array([
        get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA'),
        get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')
    ], dtype=float)
    eia_obs = np.nan_to_num(eia_obs, nan=0.0)

    # --- B. Define Strict Bounds ---
    lower_bounds = np.maximum(0.0, rep_vector)

    upper_bounds = np.zeros(8)
    for i in range(8):
        if rep_vector[i] > mod_vector[i]:
            # Reported data is higher than your estimate: allow small buffer
            upper_bounds[i] = rep_vector[i] * 1.02
        else:
            # Normal case: keep at or below your model estimate
            upper_bounds[i] = max(mod_vector[i], lower_bounds[i] + 1e-3)

    bounds = list(zip(lower_bounds, upper_bounds))

    # --- C. Initial Guess ---
    x_init = np.clip(mod_vector, lower_bounds, upper_bounds)

    # --- D. Formulate Optimization Objective ---
    scale = np.maximum(mod_vector, 10.0)  # Safe positive scale prevents div-by-zero

    def objective(x):
        # 1. Fit to your Model Estimate
        dist_to_model = np.sum(((x - mod_vector) / scale) ** 2)

        # 2. If Reported > Model: stay as close as possible to reported
        overshoot_penalty = 0.0
        for i in range(8):
            if rep_vector[i] > mod_vector[i]:
                overshoot_penalty += 15.0 * ((x[i] - rep_vector[i]) / scale[i]) ** 2

        # 3. Temporal smoothness with prior month
        smoothness_penalty = 0.0
        if t > 0:
            x_prev = np.array(optimized_results[-1][2:10], dtype=float)
            smoothness_penalty = 0.5 * np.sum(((x - x_prev) / scale) ** 2)

        # 4. EIA reconciliation if reported
        eia_penalty = 0.0
        if np.sum(eia_obs) > 0:
            eia_diff = np.dot(H_eia, x) - eia_obs
            eia_scale = np.maximum(eia_obs, 50.0)
            eia_penalty = 2.0 * np.sum((eia_diff / eia_scale) ** 2)

        return dist_to_model + overshoot_penalty + smoothness_penalty + eia_penalty

    # --- E. Solve ---
    res = minimize(
        objective,
        x0=x_init,
        method='SLSQP',
        bounds=bounds,
        options={'maxiter': 300, 'ftol': 1e-5}
    )

    if res.success and not np.any(np.isnan(res.x)):
        x_final = res.x
    else:
        # Fallback: strictly bounded initialization
        x_final = np.clip(x_init, lower_bounds, upper_bounds)

    # --- F. Safe GOR Calculation (Prevents 0/0 and NaNs) ---
    gor_final = np.zeros(4)
    for b in range(4):
        oil_vol = x_final[b]
        gas_vol = x_final[b+4]
        if oil_vol > 0.1:
            gor_final[b] = gas_vol / oil_vol
        else:
            # Retain prior month's GOR if available
            gor_final[b] = float(optimized_results[-1][10 + b]) if t > 0 else 0.0

    # Store full record
    row_record = [
        month_dt.strftime('%Y-%m'),
        t,
        *x_final.tolist(),
        *gor_final.tolist()
    ]
    optimized_results.append(row_record)

# ==========================================
# 4. FORMAT FINAL DATAFRAME
# ==========================================
output_cols = ['Month', 'Index'] + [f'Estimated_{name}' for name in state_elements]
final_results_df = pd.DataFrame(optimized_results, columns=output_cols)

# Ensure no NaNs remain in the entire DataFrame
final_results_df = final_results_df.fillna(method='ffill').fillna(0.0)

print("=== FINAL COMPLETE & NON-NAN ALLOCATION ===")
print(final_results_df.tail(6).round(1).to_string(index=False))

final_results_df.to_csv('Allocated_Production_Results.csv', index=False)

=== FINAL COMPLETE & NON-NAN ALLOCATION ===
  Month  Index  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  Estimated_Permian_Gas  Estimated_EagleFord_Gas  Estimated_Haynesville_Gas  Estimated_Other_Gas  Estimated_Permian_GOR  Estimated_EagleFord_GOR  Estimated_Haynesville_GOR  Estimated_Other_GOR
2026-01     24                 6527.0                   1129.4                       22.0                351.0                29200.0                   8150.0                    16219.5               7349.8                    4.5                      7.2                      737.2                 20.9
2026-02     25                 6662.3                   1134.7                       25.0                330.0                30150.0                   8200.0                    16319.3               7379.7                    4.5                      7.2                      652.8                 22.4
2026-03     26                 6608.5          

/tmp/ipykernel_6475/477021245.py:185: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  final_results_df = final_results_df.fillna(method='ffill').fillna(0.0)


In [10]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==========================================
# 1. LOAD AND SANITIZE THE CSV FILE
# ==========================================
df = pd.read_csv('TX2LA.csv')
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
date_col_name = df.columns[0]
df = df.sort_values(by=date_col_name).reset_index(drop=True)

df.columns = [str(c).strip().lower().replace('_', '').replace(' ', '') for c in df.columns]

def get_val(row, prefix, name):
    clean_name = f"{prefix}{name}".lower()
    val = row.get(clean_name, 0.0)
    if pd.isna(val) or val is None:
        return 0.0
    try:
        return float(val)
    except (ValueError, TypeError):
        return 0.0

regions = ['Permian', 'EagleFord', 'Haynesville', 'Other']
state_elements = (
    [f'{r}_Oil' for r in regions] +
    [f'{r}_Gas' for r in regions] +
    [f'{r}_GOR' for r in regions]
)
n_states = len(state_elements)
T = len(df)

# ==========================================
# 2. RUN EXACT MASS-BALANCE OPTIMIZATION
# ==========================================
optimized_results = []

for t in range(T):
    row = df.iloc[t]
    month_dt = row.iloc[0]

    # --- A. Read Inputs ---
    rep_oil = np.array([get_val(row, 'o', r) for r in regions], dtype=float)
    rep_gas = np.array([get_val(row, 'g', r) for r in regions], dtype=float)
    rep_vector = np.nan_to_num(np.concatenate([rep_oil, rep_gas]), nan=0.0)

    mod_oil = np.array([get_val(row, 'mo', r) for r in regions], dtype=float)
    mod_gas = np.array([get_val(row, 'mg', r) for r in regions], dtype=float)
    mod_vector = np.nan_to_num(np.concatenate([mod_oil, mod_gas]), nan=0.0)

    # State actuals (Target system mass totals)
    tx_oil, nm_oil, la_oil = get_val(row, 'o', 'TX'), get_val(row, 'o', 'NM'), get_val(row, 'o', 'LA')
    tx_gas, nm_gas, la_gas = get_val(row, 'g', 'TX'), get_val(row, 'g', 'NM'), get_val(row, 'g', 'LA')

    target_total_oil = tx_oil + nm_oil + la_oil
    target_total_gas = tx_gas + nm_gas + la_gas

    # If state totals are missing for the latest month, default to sum of your model estimates
    if target_total_oil <= 0:
        target_total_oil = np.sum(mod_oil) if np.sum(mod_oil) > 0 else (np.sum(rep_oil) if np.sum(rep_oil) > 0 else 6000.0)
    if target_total_gas <= 0:
        target_total_gas = np.sum(mod_gas) if np.sum(mod_gas) > 0 else (np.sum(rep_gas) if np.sum(rep_gas) > 0 else 35000.0)

    # Fallback model vector handling
    if np.all(mod_vector <= 0.0):
        if t > 0:
            mod_vector = np.array(optimized_results[-1][2:10], dtype=float)
        else:
            mod_vector = np.maximum(rep_vector, 10.0)

    for i in range(8):
        if mod_vector[i] <= 0.0:
            if t > 0:
                mod_vector[i] = float(optimized_results[-1][2 + i])
            else:
                mod_vector[i] = max(float(rep_vector[i]), 10.0)

    # --- B. Define Strict Bounds ---
    lower_bounds = np.maximum(0.0, rep_vector)

    # Upper bounds: Upper bound must be at least large enough to allow exact mass balance
    upper_bounds = np.zeros(8)
    for i in range(4): # Oil
        upper_bounds[i] = max(mod_vector[i], lower_bounds[i], target_total_oil)
    for i in range(4, 8): # Gas
        upper_bounds[i] = max(mod_vector[i], lower_bounds[i], target_total_gas)

    bounds = list(zip(lower_bounds, upper_bounds))

    # --- C. Hard Equality Constraints for Exact Sum Conservation ---
    constraints = [
        # Sum of 4 Basin Oils == Total 3-State Oil
        {'type': 'eq', 'fun': lambda x: np.sum(x[0:4]) - target_total_oil},
        # Sum of 4 Basin Gases == Total 3-State Gas
        {'type': 'eq', 'fun': lambda x: np.sum(x[4:8]) - target_total_gas}
    ]

    # Initial guess normalized to hit exact sum
    x_init_oil = mod_vector[0:4] * (target_total_oil / max(np.sum(mod_vector[0:4]), 1e-3))
    x_init_gas = mod_vector[4:8] * (target_total_gas / max(np.sum(mod_vector[4:8]), 1e-3))
    x_init = np.clip(np.concatenate([x_init_oil, x_init_gas]), lower_bounds, upper_bounds)

    # --- D. Objective Function ---
    scale = np.maximum(mod_vector, 10.0)

    def objective(x):
        # 1. Stay as close to your model estimates as possible
        dist_to_model = np.sum(((x - mod_vector) / scale) ** 2)

        # 2. If Reported > Model: keep it as close to reported as possible
        overshoot_penalty = 0.0
        for i in range(8):
            if rep_vector[i] > mod_vector[i]:
                overshoot_penalty += 20.0 * ((x[i] - rep_vector[i]) / scale[i]) ** 2

        # 3. Month-to-month smooth transition
        smoothness_penalty = 0.0
        if t > 0:
            x_prev = np.array(optimized_results[-1][2:10], dtype=float)
            smoothness_penalty = 0.5 * np.sum(((x - x_prev) / scale) ** 2)

        return dist_to_model + overshoot_penalty + smoothness_penalty

    # --- E. Solve with SLSQP ---
    res = minimize(
        objective,
        x0=x_init,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 500, 'ftol': 1e-6}
    )

    if res.success and not np.any(np.isnan(res.x)):
        x_final = res.x
    else:
        # Exact proportional fallback if optimization fails
        x_final = np.maximum(rep_vector, mod_vector)
        # Re-scale to ensure exact sum match
        x_final[0:4] = x_final[0:4] * (target_total_oil / np.sum(x_final[0:4]))
        x_final[4:8] = x_final[4:8] * (target_total_gas / np.sum(x_final[4:8]))

    # --- F. Calculate GOR ---
    gor_final = np.zeros(4)
    for b in range(4):
        oil_vol = x_final[b]
        gas_vol = x_final[b+4]
        if oil_vol > 0.1:
            gor_final[b] = gas_vol / oil_vol
        else:
            gor_final[b] = float(optimized_results[-1][10 + b]) if t > 0 else 0.0

    # Store results including totals for easy verification
    row_record = [
        month_dt.strftime('%Y-%m'),
        t,
        *x_final.tolist(),
        *gor_final.tolist(),
        target_total_oil, np.sum(x_final[0:4]),
        target_total_gas, np.sum(x_final[4:8])
    ]
    optimized_results.append(row_record)

# ==========================================
# 3. FORMAT FINAL DATAFRAME
# ==========================================
output_cols = (
    ['Month', 'Index'] +
    [f'Estimated_{name}' for name in state_elements] +
    ['State_Oil_Total', 'Allocated_Oil_Sum', 'State_Gas_Total', 'Allocated_Gas_Sum']
)
final_results_df = pd.DataFrame(optimized_results, columns=output_cols)

print("=== MASS CONSERVATION VERIFICATION (LAST 6 MONTHS) ===")
cols_to_show = ['Month', 'Estimated_Permian_Oil', 'Estimated_EagleFord_Oil', 'Estimated_Haynesville_Oil', 'Estimated_Other_Oil', 'State_Oil_Total', 'Allocated_Oil_Sum']
print(final_results_df[cols_to_show].tail(6).round(1).to_string(index=False))

final_results_df.to_csv('Allocated_Production_Results.csv', index=False)

=== MASS CONSERVATION VERIFICATION (LAST 6 MONTHS) ===
  Month  Estimated_Permian_Oil  Estimated_EagleFord_Oil  Estimated_Haynesville_Oil  Estimated_Other_Oil  State_Oil_Total  Allocated_Oil_Sum
2026-01                 6267.0                   1107.0                       22.0                351.0           7747.0             7747.0
2026-02                 6662.0                   1128.5                       25.0                367.5           8183.0             8183.0
2026-03                 6608.0                   1142.4                       25.0                382.6           8158.0             8158.0
2026-04                 6697.4                   1161.8                       25.0                389.7           8274.0             8274.0
2026-05                 6706.2                   1152.8                       25.1                385.9           8270.0             8270.0
2026-06                 6618.8                   1137.0                       25.0                381.2  

In [11]:
final_results_df.to_csv('Allocated_Production_Results.csv', index=False)